# OctopusNet: Red Neuronal Distribuida con Aprendizaje Local

**Componentes:**
- Módulos paralelos (homogéneos o heterogéneos) con Forward-Forward
- Nerve ring (comunicación lateral via cross-attention)
- Coordinador con atención dinámica
- Feedback top-down

In [ ]:
!nvidia-smi
import torch
import torch.nn as nn
import math
import os
from tqdm import tqdm
from dataclasses import dataclass
from typing import List
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory
CHECKPOINT_DIR = '/content/drive/MyDrive/octopusnet_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
import random
import numpy as np

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
print(f"Seed fijado: {SEED}")


## 1. Configuration

In [ ]:
@dataclass
class OctopusNetConfig:
    dataset: str = "cifar10"
    batch_size: int = 128
    num_modules: int = 4
    bottleneck_size: int = 64
    num_classes: int = 10
    homogeneous: bool = True  # True=all CNNs, False=CNN+Transformer+LSTM
    kernel_sizes: List[int] = None
    use_nerve_ring: bool = True
    use_feedback: bool = True
    ff_threshold: float = 2.0  # Initial value (adapts if adaptive=True)
    ff_adaptive_threshold: bool = True  # DEFAULT: adaptive threshold ON
    coordinator_hidden: int = 256
    epochs: int = 50
    device: str = "cuda"
    
    # Competition mechanism (GWT-inspired) - A10 experiment
    # "soft" = standard softmax (default)
    # "gumbel" = Gumbel-softmax with hard selection (more faithful to GWT)
    # "topk" = Top-K sparse attention (only K modules contribute)
    competition_type: str = "soft"
    competition_topk: int = 2  # K value for topk competition
    gumbel_tau: float = 0.5  # Temperature for Gumbel-softmax
    
    def __post_init__(self):
        if self.kernel_sizes is None:
            self.kernel_sizes = [3, 5, 7, 9]
        if self.dataset == "cifar100":
            self.num_classes = 100

print("Config ready! (Adaptive threshold ON by default)")

## 2. Modules (CNN, Transformer, LSTM)

In [ ]:
import torch.nn.functional as F

class CNNModule(nn.Module):
    def __init__(self, kernel_size=3, bottleneck_size=64, in_channels=3,
                 channels=[64, 128, 256], input_size=32, ff_threshold=2.0,
                 adaptive_threshold=True):
        super().__init__()
        self.name = f"CNN_res{input_size}"
        self.input_size = input_size
        padding = kernel_size // 2
        self.conv1 = nn.Conv2d(in_channels,  channels[0], kernel_size, padding=padding)
        self.conv2 = nn.Conv2d(channels[0],  channels[1], kernel_size, padding=padding)
        self.conv3 = nn.Conv2d(channels[1],  channels[2], kernel_size, padding=padding)
        self.relu  = nn.ReLU()
        self.pool  = nn.AdaptiveAvgPool2d((2, 2))
        self.bottleneck = nn.Linear(channels[2] * 4, bottleneck_size)
        self.threshold = ff_threshold
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        for m in [self.conv1, self.conv2, self.conv3]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def _conv_features(self, x):
        # Downsample to this module's resolution before conv
        if x.shape[-1] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size),
                              mode='bilinear', align_corners=False)
        f1 = self.relu(self.conv1(x))
        f2 = self.relu(self.conv2(F.layer_norm(f1, f1.shape[1:])))
        f3 = self.relu(self.conv3(F.layer_norm(f2, f2.shape[1:])))
        return f1, f2, f3

    def forward(self, x):
        _, _, f3 = self._conv_features(x)
        h = self.pool(f3).view(f3.size(0), -1)
        h = self.bottleneck(h)
        return h / (h.norm(dim=-1, keepdim=True) + 1e-8)

    def train_ff(self, x_pos, x_neg):
        _, _, f3p = self._conv_features(x_pos)
        _, _, f3n = self._conv_features(x_neg)
        # Goodness = mean(f3^2) global — works with Fourier+multiscale
        # (lower res modules have proportionally stronger Fourier signal)
        g_pos = (f3p ** 2).mean(dim=[1, 2, 3])
        g_neg = (f3n ** 2).mean(dim=[1, 2, 3])
        # Adaptive threshold: midpoint between pos and neg each batch
        self.threshold = (g_pos.mean().item() + g_neg.mean().item()) / 2
        loss = (torch.log(1 + torch.exp(-(g_pos - self.threshold))) +
                torch.log(1 + torch.exp( g_neg - self.threshold))).mean()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item(), g_pos.mean().item(), g_neg.mean().item()

print("CNNModule defined! (multi-scale, kernel 3x3, global goodness)")

In [ ]:
class TransformerModule(nn.Module):
    def __init__(self, bottleneck_size=64, in_channels=3, patch_size=4,
                 embed_dim=64, num_heads=4, num_layers=2, input_size=32,
                 ff_threshold=2.0, adaptive_threshold=False):
        super().__init__()
        self.name = "Transformer"
        self.num_patches = (input_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        encoder_layer    = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                                      dim_feedforward=embed_dim*4,
                                                      batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm  = nn.LayerNorm(embed_dim)
        self.bottleneck  = nn.Linear(embed_dim, bottleneck_size)
        self.threshold = ff_threshold
        self.adaptive_threshold = adaptive_threshold
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        nn.init.normal_(self.pos_embed, std=0.02)

    def _embed(self, x):
        """Patch embedding output — used for goodness (pre-transformer, raw energy)."""
        return self.patch_embed(x).flatten(2).transpose(1, 2)  # (B, N, embed_dim)

    def forward(self, x):
        expected = int(self.num_patches ** 0.5) * self.patch_embed.kernel_size[0]
        if x.shape[-1] != expected:
            x = F.interpolate(x, size=(expected, expected),
                              mode='bilinear', align_corners=False)
        h = self._embed(x) + self.pos_embed
        h = self.transformer(h)
        h = self.layer_norm(h.mean(dim=1))
        h = self.bottleneck(h)
        return h / (h.norm(dim=-1, keepdim=True) + 1e-8)

    def train_ff(self, x_pos, x_neg):
        # Goodness on patch embeddings — raw conv energy before transformer
        expected = int(self.num_patches ** 0.5) * self.patch_embed.kernel_size[0]
        if x_pos.shape[-1] != expected:
            x_pos = F.interpolate(x_pos, size=(expected, expected),
                                  mode='bilinear', align_corners=False)
            x_neg = F.interpolate(x_neg, size=(expected, expected),
                                  mode='bilinear', align_corners=False)
        e_pos = self._embed(x_pos)  # (B, N, E)
        e_neg = self._embed(x_neg)
        g_pos = (e_pos ** 2).mean(dim=[1, 2])   # mean over patches and embed dims
        g_neg = (e_neg ** 2).mean(dim=[1, 2])
        if self.adaptive_threshold:
            self.threshold = (g_pos.mean().item() + g_neg.mean().item()) / 2
        loss = (torch.log(1 + torch.exp(-(g_pos - self.threshold))) +
                torch.log(1 + torch.exp( g_neg - self.threshold))).mean()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item(), g_pos.mean().item(), g_neg.mean().item()

print("TransformerModule defined! (goodness on patch embeddings)")

In [ ]:
class LSTMModule(nn.Module):
    def __init__(self, bottleneck_size=64, in_channels=3, hidden_size=128,
                 num_layers=2, input_size=32, ff_threshold=2.0, adaptive_threshold=False):
        super().__init__()
        self.name = "LSTM"
        self.input_size = input_size
        self.lstm       = nn.LSTM(input_size=in_channels * input_size,
                                  hidden_size=hidden_size,
                                  num_layers=num_layers, batch_first=True,
                                  bidirectional=True)
        self.bottleneck = nn.Linear(hidden_size * 2, bottleneck_size)
        self.threshold = ff_threshold
        self.adaptive_threshold = adaptive_threshold
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

    def _hidden(self, x):
        """LSTM hidden state — used for goodness. RMS-norm applied to input sequence."""
        if x.shape[-1] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size),
                              mode='bilinear', align_corners=False)
        seq = x.permute(0, 2, 1, 3).reshape(x.size(0), self.input_size, -1)
        seq = seq / (seq.norm(dim=-1, keepdim=True) + 1e-8)  # normalize each scanline
        out, _ = self.lstm(seq)
        return out[:, -1, :]   # last timestep, (B, hidden*2)

    def forward(self, x):
        h = self.bottleneck(self._hidden(x))
        return h / (h.norm(dim=-1, keepdim=True) + 1e-8)

    def train_ff(self, x_pos, x_neg):
        h_pos = self._hidden(x_pos)
        h_neg = self._hidden(x_neg)
        g_pos = (h_pos ** 2).mean(dim=-1)   # mean squared hidden state
        g_neg = (h_neg ** 2).mean(dim=-1)
        if self.adaptive_threshold:
            self.threshold = (g_pos.mean().item() + g_neg.mean().item()) / 2
        loss = (torch.log(1 + torch.exp(-(g_pos - self.threshold))) +
                torch.log(1 + torch.exp( g_neg - self.threshold))).mean()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item(), g_pos.mean().item(), g_neg.mean().item()

print("LSTMModule defined! (RMS-norm on input, goodness on hidden state)")

In [ ]:
def create_modules(config):
    """Factory: crea módulos homogéneos con multi-escala [32,16,8,4]."""
    if config.dataset in ["mnist", "fashion_mnist"]:
        in_channels = 1
        base_size   = 28
    else:
        in_channels = 3
        base_size   = 32

    # Resoluciones por módulo — especialización por resolución, no por kernel
    scales = [base_size, base_size // 2, base_size // 4, max(base_size // 8, 4)]

    modules = nn.ModuleList()
    if config.homogeneous:
        for i in range(config.num_modules):
            scale = scales[i] if i < len(scales) else scales[-1]
            modules.append(CNNModule(
                kernel_size=3,                        # siempre 3x3
                bottleneck_size=config.bottleneck_size,
                in_channels=in_channels,
                input_size=scale,
                ff_threshold=config.ff_threshold,
                adaptive_threshold=True,
            ))
    else:
        # Heterogéneo: CNN 32 + CNN 16 + Transformer + LSTM
        modules.append(CNNModule(kernel_size=3, bottleneck_size=config.bottleneck_size,
                                 in_channels=in_channels, input_size=scales[0]))
        modules.append(CNNModule(kernel_size=3, bottleneck_size=config.bottleneck_size,
                                 in_channels=in_channels, input_size=scales[1]))
        modules.append(TransformerModule(bottleneck_size=config.bottleneck_size,
                                         in_channels=in_channels, input_size=scales[2],
                                         ff_threshold=config.ff_threshold))
        modules.append(LSTMModule(bottleneck_size=config.bottleneck_size,
                                  in_channels=in_channels, input_size=scales[3],
                                  ff_threshold=config.ff_threshold))

    print(f"Created {len(modules)} modules: {[m.name for m in modules]}")
    print(f"Resolutions: {[getattr(m, 'input_size', '?') for m in modules]}")
    return modules

print("create_modules ready! (multi-scale by default)")

## 3. Nerve Ring (Cross-Attention)

In [ ]:
class NerveRing(nn.Module):
    """Cross-attention between module bottlenecks."""
    
    def __init__(self, bottleneck_size=32, num_heads=4, dropout=0.1):
        super().__init__()
        self.bottleneck_size = bottleneck_size
        self.num_heads = num_heads
        self.head_dim = bottleneck_size // num_heads
        
        self.W_q = nn.Linear(bottleneck_size, bottleneck_size)
        self.W_k = nn.Linear(bottleneck_size, bottleneck_size)
        self.W_v = nn.Linear(bottleneck_size, bottleneck_size)
        self.W_o = nn.Linear(bottleneck_size, bottleneck_size)
        self.layer_norm = nn.LayerNorm(bottleneck_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, bottlenecks):
        B, N, D = bottlenecks.shape
        
        Q = self.W_q(bottlenecks).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(bottlenecks).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(bottlenecks).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, D)
        out = self.W_o(out)
        
        return self.layer_norm(bottlenecks + self.dropout(out)), attn

print("NerveRing defined!")

## 4. Coordinator

In [ ]:
import torch.nn.functional as F

class Coordinator(nn.Module):
    """Central coordinator with attention, feedback, and GWT-inspired competition."""
    
    def __init__(self, num_modules=4, bottleneck_size=32, hidden_dim=256,
                 num_classes=10, use_feedback=True, competition_type="soft",
                 competition_topk=2, gumbel_tau=0.5):
        super().__init__()
        self.num_modules = num_modules
        self.bottleneck_size = bottleneck_size
        self.use_feedback = use_feedback
        
        # Competition mechanism
        self.competition_type = competition_type
        self.competition_topk = competition_topk
        self.gumbel_tau = gumbel_tau
        
        self.W_q = nn.Linear(num_modules * bottleneck_size, bottleneck_size)
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU())
        self.classifier = nn.Linear(hidden_dim // 2, num_classes)
        
        if use_feedback:
            self.feedback_gen = nn.ModuleList([
                nn.Sequential(nn.Linear(bottleneck_size + 1, bottleneck_size), nn.Sigmoid())
                for _ in range(num_modules)])
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

    def compute_attention(self, bottlenecks):
        """Compute attention with different competition mechanisms."""
        B = bottlenecks.shape[0]
        query = self.W_q(bottlenecks.view(B, -1)).unsqueeze(1)
        scores = torch.bmm(query, bottlenecks.transpose(1, 2)).squeeze(1) / math.sqrt(self.bottleneck_size)
        
        # Apply competition mechanism
        if self.competition_type == "gumbel":
            # Gumbel-softmax: hard selection, more faithful to GWT
            alpha = F.gumbel_softmax(scores, tau=self.gumbel_tau, hard=True, dim=-1)
        elif self.competition_type == "topk":
            # Top-K: only top K modules contribute
            topk_vals, topk_idx = torch.topk(scores, k=self.competition_topk, dim=-1)
            mask = torch.zeros_like(scores)
            mask.scatter_(-1, topk_idx, 1.0)
            masked_scores = scores * mask + (1 - mask) * (-1e9)
            alpha = torch.softmax(masked_scores, dim=-1)
        else:  # "soft" (default)
            alpha = torch.softmax(scores, dim=-1)
        
        return alpha

    def forward(self, bottlenecks):
        alpha = self.compute_attention(bottlenecks)
        
        h_agg = (bottlenecks * alpha.unsqueeze(-1)).sum(dim=1)
        logits = self.classifier(self.decoder(h_agg))
        
        feedback = None
        if self.use_feedback:
            feedback = [self.feedback_gen[i](torch.cat([h_agg.detach(), alpha[:, i:i+1].detach()], -1))
                        for i in range(self.num_modules)]
        
        return logits, feedback, alpha

    def train_step(self, bottlenecks, labels):
        logits, _, _ = self.forward(bottlenecks.detach())
        loss = self.criterion(logits, labels)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item(), (logits.argmax(-1) == labels).float().mean().item()

print("Coordinator with GWT competition defined!")

## 5. OctopusNet (Full Model)

In [ ]:
class OctopusNet(nn.Module):
    """Main OctopusNet - supports homogeneous/heterogeneous modules and GWT competition."""
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Create modules (homogeneous or heterogeneous)
        self.modules_list = create_modules(config)
        self.num_modules = len(self.modules_list)
        
        # Nerve ring
        self.nerve_ring = NerveRing(config.bottleneck_size) if config.use_nerve_ring else None
        
        # Coordinator with competition mechanism
        self.coordinator = Coordinator(
            num_modules=self.num_modules,
            bottleneck_size=config.bottleneck_size,
            hidden_dim=config.coordinator_hidden,
            num_classes=config.num_classes,
            use_feedback=config.use_feedback,
            competition_type=config.competition_type,
            competition_topk=config.competition_topk,
            gumbel_tau=config.gumbel_tau)

    def forward(self, x):
        # Each module normalizes its own output; stack for nerve ring
        bottlenecks = torch.stack([m(x) for m in self.modules_list], dim=1)
        
        # Nerve ring enriches representations
        if self.nerve_ring:
            bottlenecks, _ = self.nerve_ring(bottlenecks)
        
        # Coordinator aggregates and classifies
        logits, _, _ = self.coordinator(bottlenecks)
        return logits

    def train_modules_ff(self, x_pos, x_neg):
        results = [m.train_ff(x_pos, x_neg) for m in self.modules_list]
        return [r[0] for r in results], [r[1] for r in results], [r[2] for r in results]

    def train_coordinator(self, x, labels):
        with torch.no_grad():
            bottlenecks = torch.stack([m(x) for m in self.modules_list], dim=1)
            if self.nerve_ring:
                bottlenecks, _ = self.nerve_ring(bottlenecks)
        return self.coordinator.train_step(bottlenecks, labels)

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            return self.forward(x).argmax(-1)

print("OctopusNet with GWT competition defined!")

## 6. Data & Training Utilities

In [ ]:
def get_dataloaders(config):
    if config.dataset in ["mnist", "fashion_mnist"]:
        train_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
        test_transform  = train_transform
    else:
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])
        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

    if config.dataset == "mnist":
        train_ds = datasets.MNIST('./data', train=True, download=True, transform=train_transform)
        test_ds  = datasets.MNIST('./data', train=False, download=True, transform=test_transform)
    elif config.dataset == "cifar10":
        train_ds = datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
        test_ds  = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)
    elif config.dataset == "cifar100":
        train_ds = datasets.CIFAR100('./data', train=True, download=True, transform=train_transform)
        test_ds  = datasets.CIFAR100('./data', train=False, download=True, transform=test_transform)

    return (DataLoader(train_ds, batch_size=config.batch_size, shuffle=True),
            DataLoader(test_ds,  batch_size=config.batch_size, shuffle=False))

# ── Fourier label embedding ────────────────────────────────────────────────────
_FOURIER_CACHE = {}

def _make_fourier_patterns(num_classes, height, width, device):
    key = (num_classes, height, width, str(device))
    if key in _FOURIER_CACHE:
        return _FOURIER_CACHE[key]
    orientations = [0, 45, 90, 135]
    frequencies  = [1, 2, 3]
    cy = torch.linspace(0, 2*math.pi, height, device=device)
    cx = torch.linspace(0, 2*math.pi, width,  device=device)
    gy, gx = torch.meshgrid(cy, cx, indexing='ij')
    patterns = []
    for idx in range(num_classes):
        angle = math.radians(orientations[idx % len(orientations)])
        freq  = frequencies[idx // len(orientations)]
        wave  = torch.sin(freq * (gx*math.cos(angle) + gy*math.sin(angle)))
        wave  = (wave - wave.min()) / (wave.max() - wave.min() + 1e-8)
        patterns.append(wave)
    result = torch.stack(patterns, dim=0)
    _FOURIER_CACHE[key] = result
    return result

def overlay_label(images, labels, num_classes=10, strength=0.5):
    """
    Fourier label embedding: blend sinusoidal class pattern into image.

    strength=0.5 confirmed empirically with multi-scale [32,16,8,4]:
      res=32: sep=0.07  res=16: sep=0.20  res=8: sep=0.30  res=4: sep=0.61
    Signal scales naturally with F.interpolate — lower res = stronger signal.
    """
    B, C, H, W = images.shape
    patterns = _make_fourier_patterns(num_classes, H, W, images.device)
    pat = patterns[labels].unsqueeze(1).expand(-1, C, -1, -1)
    return images * (1.0 - strength) + pat * strength

def create_negatives(images, labels, num_classes=10):
    wrong = torch.randint(0, num_classes, labels.shape, device=labels.device)
    mask = wrong == labels
    while mask.any():
        wrong[mask] = torch.randint(0, num_classes, (mask.sum(),), device=labels.device)
        mask = wrong == labels
    return overlay_label(images, wrong, num_classes)

print("Utilities ready! (Fourier strength=0.5, multi-scale compatible)")

In [ ]:
# ── Sanity check: FF health antes del diagnóstico completo ────────────────────
# Corre esto primero. Esperado: sep crece a >0.05 en 20 pasos para todos los módulos.

_cfg = OctopusNetConfig(dataset="cifar10", batch_size=128, device=DEVICE)
_train_loader, _ = get_dataloaders(_cfg)

# 4 módulos con sus resoluciones
_scales = [32, 16, 8, 4]
_modules = [CNNModule(kernel_size=3, in_channels=3, input_size=s).to(DEVICE) for s in _scales]

print(f"  {'step':>4}  " + "  ".join(f"res={s:>2} sep" for s in _scales))
for step, (imgs, labels) in enumerate(_train_loader):
    if step >= 20: break
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    xp = overlay_label(imgs, labels)
    xn = create_negatives(imgs, labels)
    results = [m.train_ff(xp, xn) for m in _modules]
    if step % 4 == 0 or step == 19:
        seps = "  ".join(f"{r[1]-r[2]:>10.4f}" for r in results)
        print(f"  {step+1:>4}  {seps}")

print()
all_ok = all(r[1]-r[2] > 0.01 for r in results)
print("OK - FF separando en todos los módulos" if all_ok else "REVISAR - algún módulo no separa")

## 7. Training Loop

In [ ]:
def train_epoch(model, loader, config):
    model.train()
    ff_losses = [0.0] * model.num_modules
    coord_loss, coord_acc, n = 0.0, 0.0, 0
    
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(config.device), labels.to(config.device)
        
        # Phase 1: FF training
        x_pos = overlay_label(imgs, labels, config.num_classes)
        x_neg = create_negatives(imgs, labels, config.num_classes)
        losses, _, _ = model.train_modules_ff(x_pos, x_neg)
        for i, l in enumerate(losses): ff_losses[i] += l
        
        # Phase 2: Coordinator
        loss, acc = model.train_coordinator(imgs, labels)
        coord_loss += loss
        coord_acc += acc
        n += 1
    
    return [l/n for l in ff_losses], coord_loss/n, coord_acc/n

def evaluate(model, loader, config):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(config.device), labels.to(config.device)
            correct += (model.predict(imgs) == labels).sum().item()
            total += len(labels)
    return correct / total

# ============ CHECKPOINT FUNCTIONS ============

def save_checkpoint(model, history, epoch, experiment_name, config):
    """Save model checkpoint to Google Drive."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'history': history,
        'config': {
            'dataset': config.dataset,
            'num_modules': config.num_modules,
            'bottleneck_size': config.bottleneck_size,
            'homogeneous': config.homogeneous,
            'competition_type': config.competition_type,
            'ff_adaptive_threshold': config.ff_adaptive_threshold,
        }
    }
    path = f'{CHECKPOINT_DIR}/{experiment_name}_epoch{epoch}.pt'
    torch.save(checkpoint, path)
    print(f"  [Checkpoint saved: {path}]")
    return path

def load_checkpoint(experiment_name, config, device):
    """Load latest checkpoint for an experiment."""
    import glob
    pattern = f'{CHECKPOINT_DIR}/{experiment_name}_epoch*.pt'
    files = glob.glob(pattern)
    
    if not files:
        print(f"No checkpoint found for {experiment_name}")
        return None, None, 0
    
    # Find latest epoch
    latest = max(files, key=lambda x: int(x.split('_epoch')[-1].replace('.pt', '')))
    print(f"Loading checkpoint: {latest}")
    
    checkpoint = torch.load(latest, map_location=device)
    
    # Recreate model
    model = OctopusNet(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    return model, checkpoint['history'], checkpoint['epoch']

def train_with_checkpoints(model, train_loader, test_loader, config, experiment_name, start_epoch=1):
    """Training loop with checkpoint saving after each epoch."""
    history = {'ff': [], 'coord_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(start_epoch, config.epochs + 1):
        ff, cl, ta = train_epoch(model, train_loader, config)
        te = evaluate(model, test_loader, config)
        
        history['ff'].append(ff)
        history['coord_loss'].append(cl)
        history['train_acc'].append(ta)
        history['test_acc'].append(te)
        
        print(f"Epoch {epoch}: Test acc = {te:.4f}")
        
        # Save checkpoint every epoch
        save_checkpoint(model, history, epoch, experiment_name, config)
    
    return history

print("Training functions with checkpoints ready!")

## 8. Diagnostic: FF Health Check

Run this **before** any long training to verify FF is learning correctly.

Answers:
1. Is g_pos > g_neg and is the gap growing? (FF health)
2. Are modules learning differently? (specialization signal)
3. Is coordinator accuracy improving? (end-to-end signal)
4. Are there NaN/Inf values? (numerical stability)
5. What is the effective receptive field problem? (kernel audit)

In [ ]:
# ── Diagnostic config ─────────────────────────────────────────────────────────
DIAG_DATASET  = "cifar10"
DIAG_EPOCHS   = 3
DIAG_MODULES  = 4
DIAG_SCALES   = [32, 16, 8, 4]   # resolución por módulo
DIAG_BATCH    = 128

# ── Kernel audit ───────────────────────────────────────────────────────────────
print("─" * 55)
print("KERNEL AUDIT (kernel 3x3, especialización por resolución)")
print("─" * 55)
for i, s in enumerate(DIAG_SCALES):
    rf = 1 + (3 - 1) * 3   # RF = 7 para 3 capas con kernel 3
    cov = rf / s * 100
    flag = "⚠ RF > IMAGE" if rf >= s else "OK"
    print(f"  Módulo {i+1}  res={s:>2}×{s:<2}  RF=7×7  coverage={cov:.0f}%  {flag}")
print()

# ── Build diagnostic model & data ──────────────────────────────────────────────
diag_config = OctopusNetConfig(
    dataset=DIAG_DATASET,
    epochs=DIAG_EPOCHS,
    batch_size=DIAG_BATCH,
    num_modules=DIAG_MODULES,
    device=DEVICE,
    ff_adaptive_threshold=True,
)

diag_train_loader, diag_test_loader = get_dataloaders(diag_config)
diag_model = OctopusNet(diag_config).to(DEVICE)

total_params = sum(p.numel() for p in diag_model.parameters())
print("─" * 55)
print("MODEL SIZE")
print("─" * 55)
for i, mod in enumerate(diag_model.modules_list):
    n = sum(p.numel() for p in mod.parameters())
    print(f"  Módulo {i+1} (res={DIAG_SCALES[i]:>2}): {n:,} params")
coord_n = sum(p.numel() for p in diag_model.coordinator.parameters())
print(f"  Coordinator       : {coord_n:,} params")
if diag_model.nerve_ring:
    nr_n = sum(p.numel() for p in diag_model.nerve_ring.parameters())
    print(f"  Nerve ring        : {nr_n:,} params")
print(f"  TOTAL             : {total_params:,} params")
print()
print("Starting 3-epoch diagnostic...")

In [ ]:
# ── Diagnostic training loop ───────────────────────────────────────────────────
diag_history = {
    'g_pos':      [[] for _ in range(DIAG_MODULES)],
    'g_neg':      [[] for _ in range(DIAG_MODULES)],
    'sep':        [[] for _ in range(DIAG_MODULES)],
    'ff_loss':    [[] for _ in range(DIAG_MODULES)],
    'coord_loss': [],
    'train_acc':  [],
    'test_acc':   [],
}

for epoch in range(1, DIAG_EPOCHS + 1):
    print()
    print("═" * 55)
    print(f"EPOCH {epoch}/{DIAG_EPOCHS}")
    print("═" * 55)

    diag_model.train()
    ep_g_pos   = [0.0] * DIAG_MODULES
    ep_g_neg   = [0.0] * DIAG_MODULES
    ep_ff_loss = [0.0] * DIAG_MODULES
    nb = 0

    for imgs, labels in tqdm(diag_train_loader, desc="FF training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        x_pos = overlay_label(imgs, labels, diag_config.num_classes)
        x_neg = create_negatives(imgs, labels, diag_config.num_classes)
        losses, g_pos, g_neg = diag_model.train_modules_ff(x_pos, x_neg)
        for i in range(DIAG_MODULES):
            ep_g_pos[i]   += g_pos[i]
            ep_g_neg[i]   += g_neg[i]
            ep_ff_loss[i] += losses[i]
        nb += 1

    print()
    print("FORWARD-FORWARD HEALTH")
    print("─" * 55)
    print(f"  {'Module':<14} {'g_pos':>7} {'g_neg':>7} {'sep':>7} {'loss':>8}  status")
    print("─" * 55)

    all_healthy = True
    for i in range(DIAG_MODULES):
        gp = ep_g_pos[i]  / nb
        gn = ep_g_neg[i]  / nb
        s  = gp - gn
        fl = ep_ff_loss[i] / nb

        diag_history['g_pos'][i].append(gp)
        diag_history['g_neg'][i].append(gn)
        diag_history['sep'][i].append(s)
        diag_history['ff_loss'][i].append(fl)

        if   s > 0.1: status = "OK - separating"
        elif s > 0:   status = "WEAK"
        else:
            status = "BAD - not separating"
            all_healthy = False

        print(f"  M{i+1} res={DIAG_SCALES[i]:>2}×{DIAG_SCALES[i]:<2}  "
              f"{gp:>7.3f} {gn:>7.3f} {s:>7.3f} {fl:>8.4f}  {status}")

    if not all_healthy:
        print("\n  ⚠ Algún módulo no separa. Revisar embedding o lr.")

    # Coordinator
    diag_model.train()
    tot_cl, tot_ca, nc = 0.0, 0.0, 0
    for imgs, labels in tqdm(diag_train_loader, desc="Coordinator"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        loss, acc = diag_model.train_coordinator(imgs, labels)
        tot_cl += loss; tot_ca += acc; nc += 1

    avg_cl = tot_cl / nc
    avg_ta = tot_ca / nc

    # Evaluation
    diag_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in diag_test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (diag_model.predict(imgs) == labels).sum().item()
            total   += labels.size(0)
    test_acc = correct / total

    diag_history['coord_loss'].append(avg_cl)
    diag_history['train_acc'].append(avg_ta)
    diag_history['test_acc'].append(test_acc)

    print()
    print("COORDINATOR")
    print("─" * 55)
    print(f"  Loss      : {avg_cl:.4f}")
    print(f"  Train acc : {avg_ta*100:.2f}%")
    print(f"  Test acc  : {test_acc*100:.2f}%")

    # Attention weights
    sample_imgs = next(iter(diag_test_loader))[0][:16].to(DEVICE)
    with torch.no_grad():
        bottlenecks = torch.stack([m(sample_imgs) for m in diag_model.modules_list], dim=1)
        if diag_model.nerve_ring:
            bottlenecks, _ = diag_model.nerve_ring(bottlenecks)
        _, _, attn = diag_model.coordinator(bottlenecks)
    attn_mean = attn.mean(dim=0)

    print()
    print("ATTENTION WEIGHTS")
    print("─" * 55)
    for i in range(DIAG_MODULES):
        bar = "█" * int(attn_mean[i].item() * 40)
        print(f"  M{i+1} res={DIAG_SCALES[i]:>2}: {attn_mean[i].item():.3f}  {bar}")

    attn_std = attn_mean.std().item()
    if attn_std < 0.02:
        print("  ⚠ Pesos uniformes — sin especialización aún")
    else:
        print(f"  OK — std={attn_std:.3f}, módulos diferenciando")

print()

In [ ]:
# ── Trend analysis & verdict ───────────────────────────────────────────────────
print("═" * 55)
print("TREND ANALYSIS")
print("═" * 55)

print()
print("Goodness separation per module across epochs:")
for i in range(DIAG_MODULES):
    seps = diag_history['sep'][i]
    trend = "↑ growing" if seps[-1] > seps[0] else "↓ flat/shrinking"
    vals = "  ".join(f"{s:.3f}" for s in seps)
    print(f"  Module {i+1}: {vals}  {trend}")

print()
print("Test accuracy across epochs:")
accs = diag_history['test_acc']
vals = "  ".join(f"{a*100:.2f}%" for a in accs)
trend = "↑ improving" if accs[-1] > accs[0] else "↓ not improving"
print(f"  {vals}  {trend}")

print()
print("═" * 55)
print("VERDICT")
print("═" * 55)

issues = []

for i in range(DIAG_MODULES):
    final_sep = diag_history['sep'][i][-1]
    if final_sep <= 0:
        issues.append(f"Module {i+1} not separating (sep={final_sep:.3f}) "
                      f"— check threshold & label embedding")
    elif final_sep < 0.05:
        issues.append(f"Module {i+1} weakly separating (sep={final_sep:.3f}) "
                      f"— try lower lr or more epochs before coordinator")

if accs[-1] <= accs[0] and DIAG_EPOCHS > 1:
    issues.append("Test accuracy not improving — coordinator may be "
                  "overfitting to poor module representations")

if not issues:
    print()
    print("  ✓ All checks passed. Safe to run full training.")
else:
    print(f"  {len(issues)} issue(s) found:")
    print()
    for issue in issues:
        print(f"  ✗ {issue}")
        print()

# ── Plot diagnostic summary ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i in range(DIAG_MODULES):
    axes[0].plot(range(1, DIAG_EPOCHS+1), diag_history['sep'][i],
                 marker='o', label=f'M{i+1} res={DIAG_SCALES[i]}')
axes[0].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[0].axhline(0.1, color='g', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('g_pos - g_neg')
axes[0].set_title('FF Separation (higher=better)')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(range(1, DIAG_EPOCHS+1), [a*100 for a in diag_history['test_acc']],
             marker='o', color='steelblue')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Coordinator Accuracy')
axes[1].grid(True)

# Attention at last epoch (reconstructed from attn_mean — use final run value)
attn_vals = [attn_mean[i].item() for i in range(DIAG_MODULES)]
axes[2].bar([f'M{i+1}\nres={DIAG_SCALES[i]}' for i in range(DIAG_MODULES)], attn_vals,
            color=['#4c72b0','#dd8452','#55a868','#c44e52'])
axes[2].set_ylabel('Mean Attention Weight')
axes[2].set_title('Module Contributions (last epoch)')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()


## 9. Run Experiment - HOMOGENEOUS (All CNNs)

In [ ]:
# Homogeneous config: 4 CNNs with different kernels
config_homo = OctopusNetConfig(
    dataset="cifar10",
    epochs=10,
    homogeneous=True,  # All CNNs
    device=DEVICE
)

model_homo = OctopusNet(config_homo).to(DEVICE)
train_loader, test_loader = get_dataloaders(config_homo)

print(f"\nHomogeneous model: {model_homo.num_modules} modules")
print(f"Modules: {[m.name for m in model_homo.modules_list]}")

In [ ]:
# Try to resume from checkpoint, otherwise start fresh
EXPERIMENT_NAME = "homogeneous_cifar10"

model_loaded, history_loaded, last_epoch = load_checkpoint(EXPERIMENT_NAME, config_homo, DEVICE)

if model_loaded is not None:
    model_homo = model_loaded
    history_homo = history_loaded
    start_epoch = last_epoch + 1
    print(f"Resuming from epoch {last_epoch}, best acc so far: {max(history_homo['test_acc']):.4f}")
else:
    history_homo = {'ff': [], 'coord_loss': [], 'train_acc': [], 'test_acc': []}
    start_epoch = 1
    print("Starting fresh training")

# Continue training with checkpoints
for epoch in range(start_epoch, config_homo.epochs + 1):
    ff, cl, ta = train_epoch(model_homo, train_loader, config_homo)
    te = evaluate(model_homo, test_loader, config_homo)
    
    history_homo['ff'].append(ff)
    history_homo['coord_loss'].append(cl)
    history_homo['train_acc'].append(ta)
    history_homo['test_acc'].append(te)
    
    print(f"Epoch {epoch}: Test acc = {te:.4f}")
    save_checkpoint(model_homo, history_homo, epoch, EXPERIMENT_NAME, config_homo)

print(f"\nBest homogeneous accuracy: {max(history_homo['test_acc']):.4f}")

## 10. Run Experiment - HETEROGENEOUS (CNN + Transformer + LSTM)

In [ ]:
# Heterogeneous config: CNN 3x3 + CNN 7x7 + Transformer + LSTM
config_hetero = OctopusNetConfig(
    dataset="cifar10",
    epochs=10,
    homogeneous=False,  # Heterogeneous!
    device=DEVICE
)

model_hetero = OctopusNet(config_hetero).to(DEVICE)

print(f"\nHeterogeneous model: {model_hetero.num_modules} modules")
print(f"Modules: {[m.name for m in model_hetero.modules_list]}")

In [ ]:
# Try to resume from checkpoint, otherwise start fresh
EXPERIMENT_NAME = "heterogeneous_cifar10"

model_loaded, history_loaded, last_epoch = load_checkpoint(EXPERIMENT_NAME, config_hetero, DEVICE)

if model_loaded is not None:
    model_hetero = model_loaded
    history_hetero = history_loaded
    start_epoch = last_epoch + 1
    print(f"Resuming from epoch {last_epoch}, best acc so far: {max(history_hetero['test_acc']):.4f}")
else:
    history_hetero = {'ff': [], 'coord_loss': [], 'train_acc': [], 'test_acc': []}
    start_epoch = 1
    print("Starting fresh training")

# Continue training with checkpoints
for epoch in range(start_epoch, config_hetero.epochs + 1):
    ff, cl, ta = train_epoch(model_hetero, train_loader, config_hetero)
    te = evaluate(model_hetero, test_loader, config_hetero)
    
    history_hetero['ff'].append(ff)
    history_hetero['coord_loss'].append(cl)
    history_hetero['train_acc'].append(ta)
    history_hetero['test_acc'].append(te)
    
    print(f"Epoch {epoch}: Test acc = {te:.4f}")
    save_checkpoint(model_hetero, history_hetero, epoch, EXPERIMENT_NAME, config_hetero)

print(f"\nBest heterogeneous accuracy: {max(history_hetero['test_acc']):.4f}")

## 11. Compare Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Test accuracy comparison
axes[0].plot(history_homo['test_acc'], label='Homogeneous (4 CNNs)', marker='o')
axes[0].plot(history_hetero['test_acc'], label='Heterogeneous (CNN+Trans+LSTM)', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Homogeneous vs Heterogeneous')
axes[0].legend()
axes[0].grid(True)

# FF losses per module (heterogeneous)
module_names = [m.name for m in model_hetero.modules_list]
for i, name in enumerate(module_names):
    axes[1].plot([ff[i] for ff in history_hetero['ff']], label=name)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('FF Loss')
axes[1].set_title('FF Loss per Module (Heterogeneous)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\n=== RESULTS ===")
print(f"Homogeneous best:   {max(history_homo['test_acc']):.4f}")
print(f"Heterogeneous best: {max(history_hetero['test_acc']):.4f}")

## 14. Experiment A6: Resilience — Module Failure

Simulate failure of 1, 2, and 3 modules by zeroing their output.
Coordinator redistributes attention automatically — no retraining needed.

Expected: graceful degradation vs ResNet which collapses to ~10% (random) on any internal failure.

In [ ]:
import itertools

# ── A6: Resiliencia ante fallos de módulos ────────────────────────────────────
config_resil = OctopusNetConfig(
    dataset='cifar10', epochs=100, bottleneck_size=64,
    homogeneous=True, device=DEVICE)
_, test_loader_resil = get_dataloaders(config_resil)

model_resil, _, _ = load_checkpoint('homogeneous_cifar10', config_resil, DEVICE)

def evaluate_with_failure(model, loader, failed_modules=[]):
    model.eval()
    originals = {}
    for idx in failed_modules:
        originals[idx] = model.modules_list[idx].forward
        def zero_forward(x, i=idx):
            return torch.zeros(x.shape[0], config_resil.bottleneck_size, device=x.device)
        model.modules_list[idx].forward = zero_forward
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (model.predict(imgs) == labels).sum().item()
            total += len(labels)
    for idx, fn in originals.items():
        model.modules_list[idx].forward = fn
    return correct / total

scales = [32, 16, 8, 4]

acc_full = evaluate_with_failure(model_resil, test_loader_resil, [])
print(f'Todos los modulos (baseline): {acc_full*100:.2f}%')
print()

print('── Fallo de 1 modulo ───────────────────────────────')
for i in range(4):
    acc = evaluate_with_failure(model_resil, test_loader_resil, [i])
    print(f'  Sin M{i+1} (res={scales[i]:>2}): {acc*100:.2f}%  ({(acc-acc_full)*100:+.2f}%)')

print()
print('── Fallo de 2 modulos ──────────────────────────────')
for combo in itertools.combinations(range(4), 2):
    acc = evaluate_with_failure(model_resil, test_loader_resil, list(combo))
    names = '+'.join(f'M{i+1}' for i in combo)
    print(f'  Sin {names}: {acc*100:.2f}%  ({(acc-acc_full)*100:+.2f}%)')

print()
print('── Fallo de 3 modulos (1 superviviente) ─────────────')
for combo in itertools.combinations(range(4), 3):
    acc = evaluate_with_failure(model_resil, test_loader_resil, list(combo))
    survivor = [i for i in range(4) if i not in combo][0]
    print(f'  Solo M{survivor+1} (res={scales[survivor]:>2}): {acc*100:.2f}%  ({(acc-acc_full)*100:+.2f}%)')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
labels_plot = ['Full\n(base)', 'Sin M1\n(32)', 'Sin M2\n(16)', 'Sin M3\n(8)', 'Sin M4\n(4)',
               'Solo M1', 'Solo M2', 'Solo M3', 'Solo M4']
accs_plot = [acc_full]
for i in range(4):
    accs_plot.append(evaluate_with_failure(model_resil, test_loader_resil, [i]))
for combo in itertools.combinations(range(4), 3):
    accs_plot.append(evaluate_with_failure(model_resil, test_loader_resil, list(combo)))
colors = ['steelblue'] + ['#dd8452']*4 + ['#c44e52']*4
ax.bar(labels_plot, [a*100 for a in accs_plot], color=colors)
ax.axhline(10, color='red', linestyle='--', alpha=0.7, label='Random (10%)')
ax.axhline(acc_full*100, color='green', linestyle='--', alpha=0.7, label='Baseline')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('A6: Graceful Degradation — OctopusNet')
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()


## 15. Experiment A6b: Module Dropout — Resilience via Training

Train coordinator with random module dropout (p=0.25). Compare resilience vs normal model.

In [ ]:
import random as pyrandom
import torch

config_resil_b = OctopusNetConfig(
    dataset='cifar10', epochs=100, bottleneck_size=64,
    homogeneous=True, device=DEVICE)
train_loader_rb, test_loader_rb = get_dataloaders(config_resil_b)
model_rb = OctopusNet(config_resil_b).to(DEVICE)

MODULE_DROP_PROB = 0.25

def train_epoch_with_module_dropout(model, loader, config, drop_prob=0.25):
    model.train()
    ff_losses = [0.0] * model.num_modules
    coord_loss, coord_acc, n = 0.0, 0.0, 0

    for imgs, labels in tqdm(loader, desc='Training+ModDrop'):
        imgs, labels = imgs.to(config.device), labels.to(config.device)

        x_pos = overlay_label(imgs, labels, config.num_classes)
        x_neg = create_negatives(imgs, labels, config.num_classes)
        losses, _, _ = model.train_modules_ff(x_pos, x_neg)
        for i, l in enumerate(losses):
            ff_losses[i] += l

        drop_idx = None
        if pyrandom.random() < drop_prob:
            drop_idx = pyrandom.randint(0, model.num_modules - 1)
            orig_fwd = model.modules_list[drop_idx].forward
            def zero_fwd(x, _i=drop_idx):
                return torch.zeros(x.shape[0], config.bottleneck_size, device=x.device)
            model.modules_list[drop_idx].forward = zero_fwd

        loss, acc = model.train_coordinator(imgs, labels)
        coord_loss += loss
        coord_acc += acc

        if drop_idx is not None:
            model.modules_list[drop_idx].forward = orig_fwd

        n += 1

    return [l/n for l in ff_losses], coord_loss/n, coord_acc/n

print('Entrenando con module dropout (100 epocas)...')
history_rb = {'test_acc': []}
best_acc = 0.0
for epoch in range(1, config_resil_b.epochs + 1):
    _, _, _ = train_epoch_with_module_dropout(model_rb, train_loader_rb, config_resil_b, MODULE_DROP_PROB)
    te = evaluate(model_rb, test_loader_rb, config_resil_b)
    history_rb['test_acc'].append(te)
    if te > best_acc:
        best_acc = te
        torch.save(model_rb.state_dict(), f'{CHECKPOINT_DIR}/moddrop_best.pt')
        print(f'  -> Checkpoint saved: {best_acc:.2%}')
    if epoch % 10 == 0:
        print(f'Epoch {epoch}: {te*100:.2f}%')

print(f'\nBest A6b: {max(history_rb["test_acc"])*100:.2f}%')

# Cargar modelo normal
model_resil = OctopusNet(config_resil_b).to(DEVICE)
model_resil.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/homogeneous_cifar10_epoch100.pt'))
model_resil.eval()

print('\n-- Comparacion de resiliencia --')
print(f'{"Escenario":<20} {"Normal":>8} {"ModDrop":>8} {"Mejora":>8}')
print('-' * 50)

acc_normal_full = evaluate_with_failure(model_resil, test_loader_rb, [])
acc_drop_full   = evaluate_with_failure(model_rb,    test_loader_rb, [])
print(f'{"Baseline":<20} {acc_normal_full*100:>7.2f}% {acc_drop_full*100:>7.2f}%  {"—":>7}')

for i in range(4):
    acc_n = evaluate_with_failure(model_resil, test_loader_rb, [i])
    acc_d = evaluate_with_failure(model_rb,    test_loader_rb, [i])
    mejora = (acc_d - acc_n) * 100
    res = [32,16,8,4][i]
    print(f'{"Sin M"+str(i+1)+" (res="+str(res)+")":<20} {acc_n*100:>7.2f}% {acc_d*100:>7.2f}% {mejora:>+7.2f}%')


## 12. Experiment A10: GWT Competition Mechanisms

Compare different competition strategies inspired by Global Workspace Theory:
- **Soft**: Standard softmax (default)
- **Gumbel**: Hard selection via Gumbel-softmax (more faithful to GWT)
- **Top-K**: Only K modules contribute (sparse attention)

In [ ]:
def run_competition_experiment(competition_type, topk=2, epochs=10):
    """Run experiment with specific competition mechanism (with checkpoints)."""
    config = OctopusNetConfig(
        dataset="cifar10",
        epochs=epochs,
        homogeneous=True,
        competition_type=competition_type,
        competition_topk=topk,
        device=DEVICE
    )
    
    exp_name = f"A10_{competition_type}" + (f"_k{topk}" if competition_type == "topk" else "")
    train_loader, test_loader = get_dataloaders(config)
    
    # Try to resume
    model, history, last_epoch = load_checkpoint(exp_name, config, DEVICE)
    
    if model is None:
        model = OctopusNet(config).to(DEVICE)
        history = {'test_acc': []}
        last_epoch = 0
    
    print(f"\n{'='*50}")
    print(f"Competition: {competition_type}" + (f" (k={topk})" if competition_type == "topk" else ""))
    if last_epoch > 0:
        print(f"Resuming from epoch {last_epoch}")
    print(f"{'='*50}")
    
    for epoch in range(last_epoch + 1, epochs + 1):
        _, _, _ = train_epoch(model, train_loader, config)
        te = evaluate(model, test_loader, config)
        history['test_acc'].append(te)
        print(f"Epoch {epoch}: Test acc = {te:.4f}")
        save_checkpoint(model, history, epoch, exp_name, config)
    
    return history

print("Competition experiment function with checkpoints ready!")

In [ ]:
# Run A10 experiments (5 epochs each for quick comparison)
EPOCHS_A10 = 5

# A10a: Soft attention (baseline)
history_soft = run_competition_experiment("soft", epochs=EPOCHS_A10)

# A10b: Gumbel-softmax (hard selection)
history_gumbel = run_competition_experiment("gumbel", epochs=EPOCHS_A10)

# A10c: Top-2 sparse
history_topk2 = run_competition_experiment("topk", topk=2, epochs=EPOCHS_A10)

# A10d: Top-1 (winner-take-all)
history_topk1 = run_competition_experiment("topk", topk=1, epochs=EPOCHS_A10)

In [ ]:
# Plot A10 results
plt.figure(figsize=(10, 5))
plt.plot(history_soft['test_acc'], label='Soft (baseline)', marker='o')
plt.plot(history_gumbel['test_acc'], label='Gumbel (hard)', marker='s')
plt.plot(history_topk2['test_acc'], label='Top-2', marker='^')
plt.plot(history_topk1['test_acc'], label='Top-1 (winner-take-all)', marker='d')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('A10: GWT Competition Mechanisms')
plt.legend()
plt.grid(True)
plt.show()

print(f"\n=== A10 RESULTS ===")
print(f"Soft (baseline):      {max(history_soft['test_acc']):.4f}")
print(f"Gumbel (hard):        {max(history_gumbel['test_acc']):.4f}")
print(f"Top-2:                {max(history_topk2['test_acc']):.4f}")
print(f"Top-1 (winner-take-all): {max(history_topk1['test_acc']):.4f}")

## 13. Experiment A11: Adaptive Threshold

Compare fixed threshold (2.0) vs adaptive threshold that auto-adjusts each batch:
- **Fixed**: θ = 2.0 (Hinton's default)
- **Adaptive**: θ = (mean(g_pos) + mean(g_neg)) / 2

In [ ]:
def run_threshold_experiment(adaptive=False, epochs=10):
    """Run experiment with fixed or adaptive threshold (with checkpoints)."""
    config = OctopusNetConfig(
        dataset="cifar10",
        epochs=epochs,
        homogeneous=True,
        ff_adaptive_threshold=adaptive,
        device=DEVICE
    )
    
    exp_name = f"A11_{'adaptive' if adaptive else 'fixed'}"
    train_loader, test_loader = get_dataloaders(config)
    
    # Try to resume
    model, history, last_epoch = load_checkpoint(exp_name, config, DEVICE)
    
    if model is None:
        model = OctopusNet(config).to(DEVICE)
        history = {'test_acc': [], 'thresholds': []}
        last_epoch = 0
    
    mode = "Adaptive" if adaptive else "Fixed (2.0)"
    print(f"\n{'='*50}")
    print(f"Threshold: {mode}")
    if last_epoch > 0:
        print(f"Resuming from epoch {last_epoch}")
    print(f"{'='*50}")
    
    for epoch in range(last_epoch + 1, epochs + 1):
        _, _, _ = train_epoch(model, train_loader, config)
        te = evaluate(model, test_loader, config)
        history['test_acc'].append(te)
        
        # Track thresholds
        thresholds = [m.threshold for m in model.modules_list]
        history['thresholds'].append(thresholds)
        
        print(f"Epoch {epoch}: Test acc = {te:.4f}, Thresholds = {[f'{t:.2f}' for t in thresholds]}")
        save_checkpoint(model, history, epoch, exp_name, config)
    
    return history

print("Threshold experiment function with checkpoints ready!")

In [ ]:
# Run A11 experiments
EPOCHS_A11 = 5

# A11a: Fixed threshold
history_fixed = run_threshold_experiment(adaptive=False, epochs=EPOCHS_A11)

# A11b: Adaptive threshold  
history_adaptive = run_threshold_experiment(adaptive=True, epochs=EPOCHS_A11)

In [ ]:
# Plot A11 results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy comparison
axes[0].plot(history_fixed['test_acc'], label='Fixed (2.0)', marker='o')
axes[0].plot(history_adaptive['test_acc'], label='Adaptive', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('A11: Fixed vs Adaptive Threshold')
axes[0].legend()
axes[0].grid(True)

# Threshold evolution (adaptive)
for i in range(4):
    thresholds = [t[i] for t in history_adaptive['thresholds']]
    axes[1].plot(thresholds, label=f'Module {i+1}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Threshold Value')
axes[1].set_title('Adaptive Threshold per Module')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\n=== A11 RESULTS ===")
print(f"Fixed (2.0):  {max(history_fixed['test_acc']):.4f}")
print(f"Adaptive:     {max(history_adaptive['test_acc']):.4f}")

## 16. Experiment A8: Nerve Ring Ablation

Compare OctopusNet with vs without nerve ring (lateral cross-attention between modules).
Both models identical except `use_nerve_ring` flag.

In [ ]:
# A8: Nerve Ring Ablation
EPOCHS_A8 = 50

config_a8_with = OctopusNetConfig(
    dataset='cifar10', epochs=EPOCHS_A8, bottleneck_size=64,
    homogeneous=True, device=DEVICE, use_nerve_ring=True)

config_a8_without = OctopusNetConfig(
    dataset='cifar10', epochs=EPOCHS_A8, bottleneck_size=64,
    homogeneous=True, device=DEVICE, use_nerve_ring=False)

train_loader_a8, test_loader_a8 = get_dataloaders(config_a8_with)

results_a8 = {}

for label, config_a8 in [('Con Nerve Ring', config_a8_with), ('Sin Nerve Ring', config_a8_without)]:
    print(f'\nEntrenando: {label}...')
    model_a8 = OctopusNet(config_a8).to(DEVICE)
    history_a8 = {'test_acc': []}
    best = 0.0

    for epoch in range(1, EPOCHS_A8 + 1):
        train_epoch(model_a8, train_loader_a8, config_a8)
        te = evaluate(model_a8, test_loader_a8, config_a8)
        history_a8['test_acc'].append(te)
        if te > best:
            best = te
            torch.save(model_a8.state_dict(),
                       f'{CHECKPOINT_DIR}/a8_{label.replace(" ", "_").lower()}.pt')
        if epoch % 10 == 0:
            print(f'  Epoch {epoch}: {te:.2%}')

    results_a8[label] = best
    print(f'{label}: {best:.2%}')

print('\n── Resultados A8: Nerve Ring Ablation ──')
for label, acc in results_a8.items():
    print(f'{label:<20} {acc*100:.2f}%')
diff = (results_a8['Con Nerve Ring'] - results_a8['Sin Nerve Ring']) * 100
print(f'\nImpacto nerve ring: {diff:+.2f}%')


## 17. Baseline CNN (Backprop)

CNN estándar entrenada con backprop. Mismos ~2M parámetros que OctopusNet.
Referencia para comparar accuracy y resiliencia.

In [ ]:
import torch
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

baseline = BaselineCNN().to(DEVICE)
params = sum(p.numel() for p in baseline.parameters())
print(f'Parametros BaselineCNN: {params:,}')

config_baseline = OctopusNetConfig(dataset='cifar10', epochs=50, device=DEVICE)
train_loader_b, test_loader_b = get_dataloaders(config_baseline)

optimizer = torch.optim.Adam(baseline.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

best_acc_baseline = 0.0
for epoch in range(1, 51):
    baseline.train()
    for imgs, labels in train_loader_b:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(baseline(imgs), labels)
        loss.backward()
        optimizer.step()
    scheduler.step()

    if epoch % 10 == 0:
        baseline.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader_b:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (baseline(imgs).argmax(1) == labels).sum().item()
                total += len(labels)
        acc = correct / total
        best_acc_baseline = max(best_acc_baseline, acc)
        print(f'Epoch {epoch}: {acc:.2%}')

torch.save(baseline.state_dict(), f'{CHECKPOINT_DIR}/baseline_cnn.pt')
print(f'\nBest Baseline CNN: {best_acc_baseline:.2%}')
print(f'OctopusNet FF:     52.50%')
print(f'Diferencia:        {(best_acc_baseline - 0.5250)*100:+.2f}%')


## 18. Experiment A15b: SFF Auxiliary Classifiers — 100% Local Learning

Cada módulo tiene un clasificador auxiliar local (SFF-style). El coordinador solo promedia/pondera predicciones — sin backprop global.

Tres métricas:
1. Accuracy de cada módulo individualmente
2. Coordinador promedio simple de predicciones locales
3. Coordinador con atención dinámica sobre predicciones locales

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AuxClassifier(nn.Module):
    def __init__(self, in_channels=256, num_classes=10):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_channels, num_classes)

    def forward(self, f3):
        x = self.pool(f3).view(f3.size(0), -1)
        return self.fc(x)

class LogitCoordinator(nn.Module):
    def __init__(self, num_modules=4, num_classes=10):
        super().__init__()
        self.attn = nn.Linear(num_modules * num_classes, num_modules)

    def forward(self, logits_list):
        stacked = torch.stack(logits_list, dim=1)
        flat = stacked.view(stacked.size(0), -1)
        weights = torch.softmax(self.attn(flat), dim=-1)
        return (stacked * weights.unsqueeze(-1)).sum(dim=1)

config_sff = OctopusNetConfig(
    dataset='cifar10', epochs=50, bottleneck_size=64,
    homogeneous=True, device=DEVICE)
train_loader_sff, test_loader_sff = get_dataloaders(config_sff)

model_sff = OctopusNet(config_sff).to(DEVICE)
aux_classifiers = nn.ModuleList([
    AuxClassifier(in_channels=256, num_classes=config_sff.num_classes).to(DEVICE)
    for _ in range(model_sff.num_modules)
])
logit_coord = LogitCoordinator(num_modules=model_sff.num_modules,
                                num_classes=config_sff.num_classes).to(DEVICE)

aux_optimizer = torch.optim.Adam(
    list(aux_classifiers.parameters()) + list(logit_coord.parameters()), lr=0.001)

criterion = nn.CrossEntropyLoss()
resolutions = [32, 16, 8, 4]

print('Entrenando A15b: SFF + Aux Classifiers (50 epocas)...')
best_acc = 0.0

for epoch in range(1, config_sff.epochs + 1):
    model_sff.train()
    aux_classifiers.train()
    logit_coord.train()

    for imgs, labels in train_loader_sff:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # FF training — optimizer interno por modulo, no backward externo
        x_pos = overlay_label(imgs, labels, config_sff.num_classes)
        x_neg = create_negatives(imgs, labels, config_sff.num_classes)
        model_sff.train_modules_ff(x_pos, x_neg)

        # Aux classifiers + logit coordinator
        aux_optimizer.zero_grad()
        logits_list = []
        loss_aux = 0.0
        for i, (module, aux_cls) in enumerate(zip(model_sff.modules_list, aux_classifiers)):
            res = resolutions[i]
            x_i = F.interpolate(imgs, size=(res, res), mode='bilinear', align_corners=False)
            _, _, f3 = module._conv_features(x_i)
            logits_i = aux_cls(f3.detach())
            logits_list.append(logits_i)
            loss_aux += criterion(logits_i, labels)

        logits_coord = logit_coord([l.detach() for l in logits_list])
        loss_coord = criterion(logits_coord, labels)
        (loss_aux + loss_coord).backward()
        aux_optimizer.step()

    if epoch % 10 == 0:
        model_sff.eval()
        aux_classifiers.eval()
        logit_coord.eval()

        module_accs = [0.0] * model_sff.num_modules
        acc_avg = 0.0
        acc_coord = 0.0
        total = 0

        with torch.no_grad():
            for imgs_t, labels_t in test_loader_sff:
                imgs_t, labels_t = imgs_t.to(DEVICE), labels_t.to(DEVICE)
                logits_t = []
                for i, (module, aux_cls) in enumerate(zip(model_sff.modules_list, aux_classifiers)):
                    res = resolutions[i]
                    x_i = F.interpolate(imgs_t, size=(res, res), mode='bilinear', align_corners=False)
                    _, _, f3 = module._conv_features(x_i)
                    logits_i = aux_cls(f3)
                    logits_t.append(logits_i)
                    module_accs[i] += (logits_i.argmax(1) == labels_t).sum().item()

                avg_logits = torch.stack(logits_t, dim=0).mean(dim=0)
                acc_avg += (avg_logits.argmax(1) == labels_t).sum().item()

                coord_logits = logit_coord(logits_t)
                acc_coord += (coord_logits.argmax(1) == labels_t).sum().item()
                total += len(labels_t)

        print(f'\nEpoch {epoch}:')
        for i in range(model_sff.num_modules):
            print(f'  M{i+1} (res={resolutions[i]:2d}): {module_accs[i]/total:.2%}')
        print(f'  Promedio simple:  {acc_avg/total:.2%}')
        print(f'  Coord atencion:   {acc_coord/total:.2%}')
        best_acc = max(best_acc, acc_coord / total)

torch.save({
    'model': model_sff.state_dict(),
    'aux': aux_classifiers.state_dict(),
    'coord': logit_coord.state_dict()
}, f'{CHECKPOINT_DIR}/a15b_sff.pt')
print(f'\nBest A15b (coord atencion): {best_acc:.2%}')


## 19. Visualización: Especialización por Módulo (A15b)

Heatmap de accuracy por clase por módulo — evidencia visual de especialización emergente.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Requiere model_sff, aux_classifiers del experimento A15b
# Si el runtime murió, cargar checkpoint primero:
# checkpoint = torch.load(f'{CHECKPOINT_DIR}/a15b_sff.pt', map_location=DEVICE)
# model_sff.load_state_dict(checkpoint['model'])
# aux_classifiers.load_state_dict(checkpoint['aux'])

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
num_classes = 10
resolutions = [32, 16, 8, 4]

model_sff.eval()
aux_classifiers.eval()

class_correct = np.zeros((4, num_classes))
class_total = np.zeros(num_classes)

with torch.no_grad():
    for imgs_t, labels_t in test_loader_sff:
        imgs_t, labels_t = imgs_t.to(DEVICE), labels_t.to(DEVICE)
        for i, (module, aux_cls) in enumerate(zip(model_sff.modules_list, aux_classifiers)):
            res = resolutions[i]
            x_i = F.interpolate(imgs_t, size=(res, res), mode='bilinear', align_corners=False)
            _, _, f3 = module._conv_features(x_i)
            preds = aux_cls(f3).argmax(1)
            for c in range(num_classes):
                mask = labels_t == c
                class_correct[i, c] += (preds[mask] == labels_t[mask]).sum().item()
        for c in range(num_classes):
            class_total[c] += (labels_t == c).sum().item()

acc_matrix = class_correct / class_total[None, :]  # (4, 10)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(acc_matrix, aspect='auto', cmap='RdYlGn', vmin=0.2, vmax=0.7)
ax.set_xticks(range(num_classes))
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.set_yticks(range(4))
ax.set_yticklabels(['M1 res=32','M2 res=16','M3 res=8','M4 res=4'])
ax.set_title('Especialización por módulo — Accuracy por clase (A15b)')
plt.colorbar(im, ax=ax)
for i in range(4):
    for j in range(num_classes):
        ax.text(j, i, f'{acc_matrix[i,j]:.0%}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('module_specialization.png', dpi=150)
plt.show()
print('Guardado: module_specialization.png')


## 20. ModuleDecoder — Decoder para Sleep Phase (A16)

Decoder CNN transpuesto por módulo. Invierte CNNModule: bottleneck → imagen soñada.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ModuleDecoder(nn.Module):
    """CNN transpuesta: bottleneck → imagen soñada a resolución del módulo.
    Usado en sleep phase (A16). detach() en output es crítico — evita loop adversarial.
    """
    def __init__(self, bottleneck_size=64, out_channels=3, out_size=32):
        super().__init__()
        self.out_size = out_size
        self.fc = nn.Linear(bottleneck_size, 64 * 2 * 2)
        layers = [nn.Unflatten(1, (64, 2, 2))]
        current = 2
        ch = 64
        while current < out_size:
            next_ch = max(ch // 2, out_channels)
            layers += [nn.ConvTranspose2d(ch, next_ch, 4, stride=2, padding=1), nn.ReLU()]
            ch = next_ch
            current *= 2
        layers += [nn.Conv2d(ch, out_channels, 1), nn.Sigmoid()]
        self.decoder = nn.Sequential(*layers)

    def forward(self, z):
        return self.decoder(self.fc(z))

print("ModuleDecoder definido.")


## 21. Experiment A16b: Sleep Phase + Fourier (sin multiscala)

Sleep phase combinada con Fourier overlay. Resolución fija 32x32.
¿Fourier y sleep se complementan o se cancelan?

In [ ]:
# A16b: Sleep + Fourier, sin multiscala
config_a16b = OctopusNetConfig(
    dataset='cifar10', epochs=20, bottleneck_size=64,
    homogeneous=True, device=DEVICE)
train_loader_a16b, test_loader_a16b = get_dataloaders(config_a16b)
model_a16b = OctopusNet(config_a16b).to(DEVICE)

decoders_a16b = nn.ModuleList([
    ModuleDecoder(bottleneck_size=64, out_channels=3, out_size=32).to(DEVICE)
    for _ in range(model_a16b.num_modules)
])
dec_opt_a16b = torch.optim.Adam(decoders_a16b.parameters(), lr=0.001)
coord_opt_a16b = torch.optim.Adam(model_a16b.coordinator.parameters(), lr=0.001)

# Preentrenar decoders con Fourier (misma distribucion que training)
print("A16b: Preentrenando decoders...")
for epoch in range(1, 6):
    for imgs, labels in train_loader_a16b:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        x_pos_pre = overlay_label(imgs, labels, config_a16b.num_classes)  # Bug3 fix
        dec_opt_a16b.zero_grad()
        loss = sum(
            F.mse_loss(dec(module(x_pos_pre).detach()), x_pos_pre)
            for module, dec in zip(model_a16b.modules_list, decoders_a16b)
        )
        loss.backward()
        dec_opt_a16b.step()
print("  Decoders preentrenados.")

print("A16b: Entrenando Sleep + Fourier (20 epocas)...")
criterion = nn.CrossEntropyLoss()
best_acc_a16b = 0.0

for epoch in range(1, config_a16b.epochs + 1):
    model_a16b.train()
    for dec in decoders_a16b: dec.eval()  # decoders fijos durante FF

    for imgs, labels in train_loader_a16b:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # Positivos: Fourier overlay
        x_pos = overlay_label(imgs, labels, config_a16b.num_classes)
        # Negativos dia: Fourier con label incorrecto
        x_neg_fourier = create_negatives(imgs, labels, config_a16b.num_classes)

        # FF dia: Fourier normal
        model_a16b.train_modules_ff(x_pos, x_neg_fourier)

        # FF noche: cada modulo entrena con SU propio sueno (Bug2 fix)
        for i, (module, dec) in enumerate(zip(model_a16b.modules_list, decoders_a16b)):
            noise = torch.randn(imgs.size(0), 64, device=DEVICE)
            x_dream = dec(noise).detach()
            module.train_ff(x_pos, x_dream)

        # Coordinator con imgs crudas (Bug1+4 fix)
        coord_opt_a16b.zero_grad()
        with torch.no_grad():
            bottlenecks = torch.stack([m(imgs) for m in model_a16b.modules_list], dim=1)
        result = model_a16b.coordinator(bottlenecks)  # Bug1 fix
        logits = result[0]
        loss_c = criterion(logits, labels)
        loss_c.backward()
        coord_opt_a16b.step()

    if epoch % 5 == 0:
        model_a16b.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs_t, labels_t in test_loader_a16b:
                imgs_t, labels_t = imgs_t.to(DEVICE), labels_t.to(DEVICE)
                correct += (model_a16b(imgs_t).argmax(1) == labels_t).sum().item()
                total += len(labels_t)
        acc = correct / total
        best_acc_a16b = max(best_acc_a16b, acc)
        print(f"  Epoch {epoch}: {acc:.2%}")

print(f"\nA16b Best: {best_acc_a16b:.2%}")
torch.save({'model': model_a16b.state_dict(), 'decoders': decoders_a16b.state_dict()},
           f'{CHECKPOINT_DIR}/a16b_sleep_fourier.pt')


## 22. Experiment A16c: Sleep Phase + Fourier + Multiscala (configuración completa)

Sleep phase con la configuración completa de OctopusNet: Fourier + multiscala [32,16,8,4].
Cada módulo tiene su propio decoder a su resolución específica.
Pregunta clave: ¿qué sueña cada módulo? ¿Refleja su especialización?

In [ ]:
# A16c: Sleep + Fourier + Multiscala — configuracion completa
resolutions = [32, 16, 8, 4]

config_a16c = OctopusNetConfig(
    dataset='cifar10', epochs=30, bottleneck_size=64,
    homogeneous=True, device=DEVICE)
config_a16c.use_multiscale = True
config_a16c.input_scales = [32, 16, 8, 4]
train_loader_a16c, test_loader_a16c = get_dataloaders(config_a16c)
model_a16c = OctopusNet(config_a16c).to(DEVICE)

# Decoder por modulo, cada uno a SU resolucion
decoders_a16c = nn.ModuleList([
    ModuleDecoder(bottleneck_size=64, out_channels=3, out_size=res).to(DEVICE)
    for res in resolutions
])
dec_opt_a16c = torch.optim.Adam(decoders_a16c.parameters(), lr=0.001)
coord_opt_a16c = torch.optim.Adam(model_a16c.coordinator.parameters(), lr=0.001)

# Preentrenar decoders a su resolucion con Fourier (Bug3 fix)
print("A16c: Preentrenando decoders por resolucion...")
for epoch in range(1, 6):
    for imgs, labels in train_loader_a16c:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        x_pos_pre = overlay_label(imgs, labels, config_a16c.num_classes)
        dec_opt_a16c.zero_grad()
        loss = 0
        for i, (module, dec, res) in enumerate(zip(model_a16c.modules_list, decoders_a16c, resolutions)):
            x_i = F.interpolate(x_pos_pre, size=(res, res), mode='bilinear', align_corners=False)
            z = module(x_i).detach()
            x_recon = dec(z)
            loss += F.mse_loss(x_recon, x_i)
        loss.backward()
        dec_opt_a16c.step()
print("  Decoders preentrenados.")

print("A16c: Entrenando Sleep + Fourier + Multiscala (30 epocas)...")
criterion = nn.CrossEntropyLoss()
best_acc_a16c = 0.0

for epoch in range(1, config_a16c.epochs + 1):
    model_a16c.train()
    for dec in decoders_a16c: dec.eval()

    for imgs, labels in train_loader_a16c:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # FF dia: Fourier + multiscala normal
        x_pos = overlay_label(imgs, labels, config_a16c.num_classes)
        x_neg_fourier = create_negatives(imgs, labels, config_a16c.num_classes)
        model_a16c.train_modules_ff(x_pos, x_neg_fourier)

        # FF noche: cada modulo suena desde SU resolucion (Bug2 fix)
        for i, (module, dec, res) in enumerate(zip(model_a16c.modules_list, decoders_a16c, resolutions)):
            x_i = F.interpolate(x_pos, size=(res, res), mode='bilinear', align_corners=False)
            noise = torch.randn(imgs.size(0), 64, device=DEVICE)
            x_dream = dec(noise).detach()
            module.train_ff(x_i, x_dream)

        # Coordinator (Bug1 fix)
        coord_opt_a16c.zero_grad()
        with torch.no_grad():
            bottlenecks = torch.stack(
                [m(F.interpolate(imgs, size=(res, res), mode='bilinear', align_corners=False))
                 for m, res in zip(model_a16c.modules_list, resolutions)], dim=1)
        result = model_a16c.coordinator(bottlenecks)
        logits = result[0]
        loss_c = criterion(logits, labels)
        loss_c.backward()
        coord_opt_a16c.step()

    if epoch % 5 == 0:
        model_a16c.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs_t, labels_t in test_loader_a16c:
                imgs_t, labels_t = imgs_t.to(DEVICE), labels_t.to(DEVICE)
                correct += (model_a16c(imgs_t).argmax(1) == labels_t).sum().item()
                total += len(labels_t)
        acc = correct / total
        best_acc_a16c = max(best_acc_a16c, acc)
        print(f"  Epoch {epoch}: {acc:.2%}")

print(f"\nA16c Best: {best_acc_a16c:.2%}")
torch.save({'model': model_a16c.state_dict(), 'decoders': decoders_a16c.state_dict()},
           f'{CHECKPOINT_DIR}/a16c_sleep_full.pt')

# Visualizacion de suenos por modulo
print("\nVisualizando suenos por modulo...")
model_a16c.eval()
for dec in decoders_a16c: dec.eval()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
module_names = ['M1 res=32\n(texturas finas)', 'M2 res=16\n(formas medias)',
                'M3 res=8\n(patrones gruesos)', 'M4 res=4\n(contexto global)']

with torch.no_grad():
    for i, (dec, res) in enumerate(zip(decoders_a16c, resolutions)):
        for j in range(8):
            noise = torch.randn(1, 64, device=DEVICE)
            dream = dec(noise)
            dream_32 = F.interpolate(dream, size=(32, 32), mode='bilinear', align_corners=False)
            img = dream_32[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
        axes[i, 0].set_ylabel(module_names[i], fontsize=9, rotation=90, labelpad=50)

plt.suptitle('Suenos por modulo — OctopusNet A16c (Sleep + Fourier + Multiscala)', fontsize=12)
plt.tight_layout()
plt.savefig('sleep_dreams_a16c.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: sleep_dreams_a16c.png")

# Resumen comparativo
print("\n=== Resumen A16 Sleep Phase ===")
print(f"A16b (sleep + Fourier):         {best_acc_a16b:.2%}")
print(f"A16c (sleep + Fourier + multi): {best_acc_a16c:.2%}")
print(f"Baseline (A15b, sin sleep):     53.16%")


## 23. Experiment A17: Iterative Nerve Ring

**Hypothesis**: multiple rounds of cross-attention (negotiation) improve module specialization over single-round competition.

**Control**: same base checkpoint (epoch 10), rounds in {1, 2, 3}.

**Metrics**:
- Accuracy after fine-tune
- Cosine divergence: how different module bottlenecks are (higher = more specialized)
- Attention entropy: coordinator confidence (lower = sharper)
- Round delta: how much each extra round changes representations

In [ ]:
# A17 — Isolated test: IterativeNerveRing
# Everything defined here, no changes to base architecture
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import copy


class IterativeNerveRing(nn.Module):
    """Nerve ring with N negotiation rounds instead of 1."""

    def __init__(self, bottleneck_dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=bottleneck_dim, num_heads=num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_dim)

    def forward(self, bottlenecks, num_rounds=1):
        # bottlenecks: list of (B, D) tensors
        h = torch.stack(bottlenecks, dim=1)  # (B, N, D)
        round_deltas = []
        attn_last = None
        for _ in range(num_rounds):
            h_prev = h.detach().clone()
            h_attn, attn_last = self.attn(h, h, h)
            h = self.norm(h + h_attn)
            delta = (h.detach() - h_prev).norm(dim=-1).mean().item()
            round_deltas.append(delta)
        return h, attn_last, round_deltas


def cosine_divergence(h):
    """Mean pairwise cosine distance between module bottlenecks. Higher = more specialized."""
    N = h.shape[1]
    h_norm = F.normalize(h, dim=-1)
    sim = torch.bmm(h_norm, h_norm.transpose(1, 2))  # (B, N, N)
    mask = ~torch.eye(N, dtype=torch.bool, device=h.device).unsqueeze(0).expand_as(sim)
    return (1 - sim[mask]).mean().item()


def attn_entropy(attn_weights):
    """Shannon entropy of attention. Lower = coordinator more decisive."""
    p = attn_weights.mean(dim=1).clamp(min=1e-9)  # (B, N)
    return -(p * p.log()).sum(dim=-1).mean().item()


print("IterativeNerveRing + metrics defined.")

In [ ]:
# Train base checkpoint (10 epochs) — shared across all rounds
import os

A17_CHECKPOINT = "/tmp/a17_base_epoch10.pt"

if not os.path.exists(A17_CHECKPOINT):
    print("Training base checkpoint (10 epochs)...")
    cfg_a17_base = OctopusNetConfig(
        dataset="cifar10",
        epochs=10,
        batch_size=128,
        bottleneck_size=64,
        )
    model_a17_base = OctopusNet(cfg_a17_base)
    for epoch in range(cfg_a17_base.epochs):
        train_epoch(model_a17_base, get_dataloaders(cfg_a17_base)[0], cfg_a17_base)
    torch.save(model_a17_base.state_dict(), A17_CHECKPOINT)
    print(f"Saved: {A17_CHECKPOINT}")
else:
    print(f"Checkpoint already exists: {A17_CHECKPOINT}")

In [ ]:
# A17: run rounds=1,2,3 from same checkpoint, collect metrics
FINETUNE_EPOCHS_A17 = 10
ROUNDS_LIST = [1, 2, 3]

a17_results = {}

cfg_a17 = OctopusNetConfig(
    dataset="cifar10", epochs=FINETUNE_EPOCHS_A17,
    batch_size=128, bottleneck_size=64,
)
train_loader_a17, val_loader_a17 = get_dataloaders(cfg_a17)
device_a17 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion_a17 = nn.CrossEntropyLoss()

for rounds in ROUNDS_LIST:
    print(f"\n=== rounds={rounds} ===")

    model_r = OctopusNet(cfg_a17).to(device_a17)
    model_r.load_state_dict(torch.load(A17_CHECKPOINT, map_location=device_a17))

    iter_ring = IterativeNerveRing(bottleneck_dim=cfg_a17.bottleneck_size).to(device_a17)
    opt_r = torch.optim.Adam(
        list(model_r.coordinator.parameters()) + list(iter_ring.parameters()), lr=1e-3
    )

    # Fine-tune coordinator + iter_ring (modules frozen)
    for epoch in range(FINETUNE_EPOCHS_A17):
        model_r.train(); iter_ring.train()
        for imgs, labels in train_loader_a17:
            imgs, labels = imgs.to(device_a17), labels.to(device_a17)
            opt_r.zero_grad()
            with torch.no_grad():
                bns = [mod(imgs) for mod in model_r.modules_list]
            h, _, _ = iter_ring(bns, num_rounds=rounds)
            logits, _, _ = model_r.coordinator(h)
            loss = criterion_a17(logits, labels)
            loss.backward()
            opt_r.step()

    # Eval accuracy
    model_r.eval(); iter_ring.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a17:
            imgs, labels = imgs.to(device_a17), labels.to(device_a17)
            bns = [mod(imgs) for mod in model_r.modules_list]
            h, _, _ = iter_ring(bns, num_rounds=rounds)
            logits, _, _ = model_r.coordinator(h)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    acc = correct / total

    # Metrics on val batch
    imgs_v, _ = next(iter(val_loader_a17))
    imgs_v = imgs_v.to(device_a17)
    with torch.no_grad():
        bns_v = [mod(imgs_v) for mod in model_r.modules_list]
        h_v, attn_w, deltas = iter_ring(bns_v, num_rounds=rounds)
    div = cosine_divergence(h_v)
    ent = attn_entropy(attn_w)

    a17_results[rounds] = {"acc": acc, "divergence": div, "entropy": ent, "deltas": deltas}
    print(f"  acc={acc:.4f}  divergence={div:.4f}  entropy={ent:.4f}  deltas={[round(d,4) for d in deltas]}")

print("\nDone. Run next cell for plots.")

In [ ]:
# A17 — 4-panel results plot
import matplotlib.pyplot as plt

rounds_list = sorted(a17_results.keys())
accs        = [a17_results[r]["acc"] * 100 for r in rounds_list]
divs        = [a17_results[r]["divergence"] for r in rounds_list]
ents        = [a17_results[r]["entropy"] for r in rounds_list]
last_deltas = [a17_results[r]["deltas"][-1] for r in rounds_list]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
fig.suptitle("A17: Iterative Nerve Ring", fontsize=14, fontweight="bold")

colors = ["#3b82f6", "#22c55e", "#f97316", "#a855f7"]

def bar_panel(ax, vals, title, ylabel, color):
    ax.bar(rounds_list, vals, color=color, alpha=0.85, width=0.5)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Rounds")
    ax.set_ylabel(ylabel)
    ax.set_xticks(rounds_list)
    for x, v in zip(rounds_list, vals):
        ax.text(x, v + max(vals) * 0.01, f"{v:.3f}", ha="center", fontsize=9)

bar_panel(axes[0, 0], accs,        "Accuracy (%)",       "%",    colors[0])
bar_panel(axes[0, 1], divs,        "Cosine Divergence",  "div",  colors[1])
bar_panel(axes[1, 0], ents,        "Attention Entropy",  "bits", colors[2])
bar_panel(axes[1, 1], last_deltas, "Last Round Delta",   "Δ",    colors[3])

plt.tight_layout()
plt.show()
print("Hypothesis: if acc peaks at rounds>1 AND divergence increases AND entropy drops -> negotiation helps")

## 24. Experiment A18: Channel Grouping (Ortiz Torres et al.)

Based on arXiv:2504.21662. Divide CNN channels into J groups (one per class).
Each group computes its own goodness score — no external negatives needed.

**Key idea**: instead of comparing positive vs negative images,
each channel group learns to maximize goodness for its target class and minimize for others.

**Test**: replace CNNModule with ChannelGroupCNNModule, train standalone, compare accuracy.

In [ ]:
# A18 — Isolated test: Channel Grouping
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES = 10  # CIFAR-10


class ChannelGroupCNNModule(nn.Module):
    """
    CNN module with channel grouping (Ortiz Torres et al., 2025).
    Total channels = groups * channels_per_group.
    Each group specializes in one class.
    No negative samples needed: goodness is computed per group.
    """

    def __init__(self, num_classes=10, channels_per_group=16, kernel_size=3,
                 in_channels=3, bottleneck_size=64):
        super().__init__()
        self.num_classes = num_classes
        self.cpg = channels_per_group
        total_ch = num_classes * channels_per_group

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, total_ch, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.Conv2d(total_ch, total_ch, kernel_size, padding=kernel_size // 2, groups=num_classes),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.proj = nn.Linear(total_ch * 16, bottleneck_size)

    def goodness_per_group(self, x):
        """Return goodness score for each class group. Shape: (B, num_classes)"""
        feat = self.conv(x)  # (B, total_ch, 4, 4)
        B, C, H, W = feat.shape
        feat_flat = feat.view(B, self.num_classes, self.cpg, H * W)
        # goodness = mean squared activation per group
        goodness = feat_flat.pow(2).mean(dim=[2, 3])  # (B, num_classes)
        return goodness

    def forward(self, x):
        feat = self.conv(x)
        h = self.proj(feat.flatten(1))
        return h

    def ff_loss(self, x_pos, x_neg, labels, threshold=2.0):
        """
        Channel-group FF loss.
        Positive: goodness[label] should be high.
        Negative: goodness[wrong_label] should be low.
        No separate negative image needed.
        """
        g_pos = self.goodness_per_group(x_pos)  # (B, C)
        # Select goodness for the correct class
        g_target = g_pos[torch.arange(len(labels)), labels]  # (B,)
        # For negatives: average goodness of all wrong classes
        mask = torch.ones_like(g_pos, dtype=torch.bool)
        mask[torch.arange(len(labels)), labels] = False
        g_wrong = g_pos[mask].view(len(labels), self.num_classes - 1).mean(dim=1)

        loss_pos = F.softplus(-g_target + threshold).mean()
        loss_neg = F.softplus(g_wrong - threshold).mean()
        return loss_pos + loss_neg


print("ChannelGroupCNNModule defined.")

In [ ]:
# A18: train standalone channel-group module and evaluate
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import MultiStepLR

EPOCHS_A18 = 20
device_a18 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use same dataloaders (CIFAR-10)
cfg_a18 = OctopusNetConfig(
    dataset="cifar10",
    epochs=EPOCHS_A18,
    batch_size=128,
    bottleneck_size=64,
)
train_loader_a18, val_loader_a18 = get_dataloaders(cfg_a18)

# Channel-group module + linear head on top
cg_module = ChannelGroupCNNModule(
    num_classes=10,
    channels_per_group=16,
    kernel_size=3,
    bottleneck_size=64,
).to(device_a18)

head = nn.Linear(64, 10).to(device_a18)

# Two-phase training (Ortiz Torres): local FF first, then head
# Phase 1: local FF loss
opt_ff = torch.optim.Adam(cg_module.parameters(), lr=1e-3)
sched_ff = MultiStepLR(opt_ff, milestones=[5, 12, 17], gamma=0.1)

a18_train_losses = []

for epoch in range(EPOCHS_A18):
    cg_module.train()
    total_loss = 0.0
    for imgs, labels in train_loader_a18:
        imgs, labels = imgs.to(device_a18), labels.to(device_a18)
        opt_ff.zero_grad()
        loss = cg_module.ff_loss(imgs, None, labels)
        loss.backward()
        opt_ff.step()
        total_loss += loss.item()
    sched_ff.step()
    a18_train_losses.append(total_loss / len(train_loader_a18))
    print(f"  [FF phase] epoch {epoch+1}/{EPOCHS_A18}  loss={a18_train_losses[-1]:.4f}")

# Phase 2: train linear head (frozen module)
cg_module.eval()
for p in cg_module.parameters():
    p.requires_grad_(False)

opt_head = torch.optim.Adam(head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    head.train()
    for imgs, labels in train_loader_a18:
        imgs, labels = imgs.to(device_a18), labels.to(device_a18)
        with torch.no_grad():
            h = cg_module(imgs)
        opt_head.zero_grad()
        loss = criterion(head(h), labels)
        loss.backward()
        opt_head.step()

# Evaluate
cg_module.eval()
head.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in val_loader_a18:
        imgs, labels = imgs.to(device_a18), labels.to(device_a18)
        preds = head(cg_module(imgs)).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

a18_acc = correct / total
print(f"\nA18 Channel Grouping accuracy: {a18_acc*100:.2f}%")
print(f"(baseline single module ~40-45%, OctopusNet ensemble ~53%)")

In [ ]:
# A18 — Training loss curve + goodness visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("A18: Channel Grouping (Ortiz Torres et al.)", fontsize=13, fontweight="bold")

# Loss curve
axes[0].plot(range(1, len(a18_train_losses) + 1), a18_train_losses, color="#3b82f6", linewidth=2)
axes[0].set_title("FF Loss (channel-group)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

# Goodness per class on val batch
cg_module.eval()
imgs_sample, labels_sample = next(iter(val_loader_a18))
imgs_sample = imgs_sample.to(device_a18)
with torch.no_grad():
    g = cg_module.goodness_per_group(imgs_sample).cpu().numpy()  # (B, 10)

mean_g = g.mean(axis=0)
classes = ["airplane", "auto", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
axes[1].bar(range(10), mean_g, color="#22c55e", alpha=0.85)
axes[1].set_title("Mean Goodness per Class Group")
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Goodness")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final accuracy: {a18_acc*100:.2f}%")

## 25. Experiment A18b: OctopusNet + Channel Grouping

Integra channel grouping a los 4 módulos CNN de OctopusNet.
Sin Fourier, sin multiscala — solo lo que propone Ortiz Torres.

**Pregunta**: ¿supera el 53.16% del mejor modo actual (SFF)?

**Comparación**:
- A18 (módulo solo + CG): 49.22%
- OctopusNet baseline (4 módulos, FF estándar): 52.75%
- OctopusNet SFF: 53.16%
- **A18b (OctopusNet + CG)**: ?

In [ ]:
# A18b — CNNModule con channel grouping integrado
import torch
import torch.nn as nn
import torch.nn.functional as F


class CGCNNModule(nn.Module):
    """
    CNN module con channel grouping (Ortiz Torres et al.).
    Drop-in replacement de CNNModule para OctopusNet.
    """

    def __init__(self, num_classes=10, channels_per_group=16,
                 kernel_size=3, in_channels=3, bottleneck_size=64):
        super().__init__()
        self.num_classes = num_classes
        self.cpg = channels_per_group
        total_ch = num_classes * channels_per_group  # 160

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, total_ch, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.Conv2d(total_ch, total_ch, kernel_size, padding=kernel_size // 2, groups=num_classes),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.proj = nn.Linear(total_ch * 16, bottleneck_size)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

    def goodness_per_group(self, x):
        feat = self.conv(x)
        B, C, H, W = feat.shape
        feat_flat = feat.view(B, self.num_classes, self.cpg, H * W)
        return feat_flat.pow(2).mean(dim=[2, 3])  # (B, num_classes)

    def ff_loss(self, x, labels, threshold=2.0):
        g = self.goodness_per_group(x)  # (B, C)
        g_target = g[torch.arange(len(labels)), labels]
        mask = torch.ones_like(g, dtype=torch.bool)
        mask[torch.arange(len(labels)), labels] = False
        g_wrong = g[mask].view(len(labels), self.num_classes - 1).mean(dim=1)
        return F.softplus(-g_target + threshold).mean() + F.softplus(g_wrong - threshold).mean()


print("CGCNNModule defined.")

In [ ]:
# A18b — OctopusNet con 4 CGCNNModules
# Sin multiscala, sin Fourier — configuracion limpia Ortiz Torres
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import MultiStepLR

EPOCHS_A18B = 30
KERNEL_SIZES_A18B = [3, 5, 7, 9]  # heterogeneous kernels, mismos que baseline
device_a18b = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg_a18b = OctopusNetConfig(
    dataset="cifar10",
    epochs=EPOCHS_A18B,
    batch_size=128,
    bottleneck_size=64,
    use_nerve_ring=True,
)
train_loader_a18b, val_loader_a18b = get_dataloaders(cfg_a18b)

# Build 4 CG modules with different kernels
cg_modules = nn.ModuleList([
    CGCNNModule(kernel_size=k, bottleneck_size=64).to(device_a18b)
    for k in KERNEL_SIZES_A18B
])

# Nerve ring + coordinator from OctopusNet (reuse classes already defined)
nerve_ring_a18b = NerveRing(bottleneck_size=64).to(device_a18b)

coordinator_a18b = Coordinator(num_modules=4, bottleneck_size=64, num_classes=10).to(device_a18b)

# Optimizers
opt_modules = torch.optim.Adam(cg_modules.parameters(), lr=1e-3)
opt_coord   = torch.optim.Adam(
    list(nerve_ring_a18b.parameters()) + list(coordinator_a18b.parameters()),
    lr=1e-3
)
sched_mod  = MultiStepLR(opt_modules, milestones=[10, 20, 27], gamma=0.1)
sched_coord = MultiStepLR(opt_coord,  milestones=[10, 20, 27], gamma=0.1)

criterion = nn.CrossEntropyLoss()
a18b_history = {"train_loss": [], "val_acc": []}

for epoch in range(EPOCHS_A18B):
    # ── Train ──
    for mod in cg_modules: mod.train()
    nerve_ring_a18b.train()
    coordinator_a18b.train()

    total_loss = 0.0
    for imgs, labels in train_loader_a18b:
        imgs, labels = imgs.to(device_a18b), labels.to(device_a18b)

        # Local FF loss per module
        opt_modules.zero_grad()
        ff_loss = sum(mod.ff_loss(imgs, labels) for mod in cg_modules)
        ff_loss.backward()
        opt_modules.step()

        # Coordinator loss (backprop only through coordinator)
        opt_coord.zero_grad()
        with torch.no_grad():
            bottlenecks = [mod(imgs) for mod in cg_modules]
        h_ring = nerve_ring_a18b(torch.stack(bottlenecks, dim=1))
        logits = coordinator_a18b(h_ring)
        coord_loss = criterion(logits, labels)
        coord_loss.backward()
        opt_coord.step()

        total_loss += coord_loss.item()

    sched_mod.step()
    sched_coord.step()

    # ── Val ──
    for mod in cg_modules: mod.eval()
    nerve_ring_a18b.eval()
    coordinator_a18b.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a18b:
            imgs, labels = imgs.to(device_a18b), labels.to(device_a18b)
            bottlenecks = [mod(imgs) for mod in cg_modules]
            h_ring = nerve_ring_a18b(torch.stack(bottlenecks, dim=1))
            preds = coordinator_a18b(h_ring).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    avg_loss = total_loss / len(train_loader_a18b)
    a18b_history["train_loss"].append(avg_loss)
    a18b_history["val_acc"].append(acc)
    print(f"  epoch {epoch+1}/{EPOCHS_A18B}  loss={avg_loss:.4f}  val_acc={acc:.4f}")

best_a18b = max(a18b_history["val_acc"])
print(f"\nA18b best val_acc: {best_a18b*100:.2f}%")
print(f"Baseline OctopusNet: 52.75% | SFF: 53.16%")

In [ ]:
# A18b — resultados
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("A18b: OctopusNet + Channel Grouping", fontsize=13, fontweight="bold")

epochs_range = range(1, len(a18b_history["train_loss"]) + 1)

axes[0].plot(epochs_range, a18b_history["train_loss"], color="#3b82f6", linewidth=2)
axes[0].set_title("Coordinator Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, [v * 100 for v in a18b_history["val_acc"]],
             color="#22c55e", linewidth=2)
axes[1].axhline(52.75, color="#f97316", linestyle="--", label="Baseline 52.75%")
axes[1].axhline(53.16, color="#ef4444", linestyle="--", label="SFF 53.16%")
axes[1].set_title("Val Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("%")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"A18b: {best_a18b*100:.2f}%  |  Baseline: 52.75%  |  SFF: 53.16%")

## 26. Experiment A6b: Module Dropout + Channel Grouping

Misma arquitectura que A18b pero el coordinador se entrena con module dropout (p=0.5).

**Hipótesis**: Module Dropout fuerza al coordinador a no depender de ningún módulo único → sube el piso de resiliencia.

**Comparación**:
- A18b (sin ModDrop): 64.17%, floor 41.47%
- A6b (con ModDrop): ?

In [ ]:
# A6b — Setup: CGCNNModule + Module Dropout training
import torch, torch.nn as nn, torch.nn.functional as F, random
from torch.optim.lr_scheduler import MultiStepLR
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/octopusnet_checkpoints"
device_a6b = torch.device("cuda" if torch.cuda.is_available() else "cpu")

KERNEL_SIZES_A6B = [3, 5, 7, 9]
EPOCHS_A6B = 30

cfg_a6b = OctopusNetConfig(
    dataset="cifar10",
    epochs=EPOCHS_A6B,
    batch_size=128,
    bottleneck_size=64,
    use_nerve_ring=True,
)
train_loader_a6b, val_loader_a6b = get_dataloaders(cfg_a6b)

cg_modules_md = nn.ModuleList([
    CGCNNModule(kernel_size=k, bottleneck_size=64).to(device_a6b)
    for k in KERNEL_SIZES_A6B
])
nerve_ring_md  = NerveRing(bottleneck_size=64).to(device_a6b)
coordinator_md = Coordinator(num_modules=4, bottleneck_size=64, num_classes=10).to(device_a6b)

print("A6b models defined.")

In [ ]:
# A6b — Training loop con Module Dropout
# Si existe checkpoint, carga y continua desde epoch guardada
import os

opt_mod_md   = torch.optim.Adam(cg_modules_md.parameters(), lr=1e-3)
opt_coord_md = torch.optim.Adam(
    list(nerve_ring_md.parameters()) + list(coordinator_md.parameters()), lr=1e-3
)
sched_mod_md   = MultiStepLR(opt_mod_md,   milestones=[10, 20, 27], gamma=0.1)
sched_coord_md = MultiStepLR(opt_coord_md, milestones=[10, 20, 27], gamma=0.1)
criterion_md = nn.CrossEntropyLoss()
history_a6b  = {"test_acc": []}
start_epoch  = 0

ckpt_path = f"{SAVE_DIR}/a6b_epoch24.pt"
if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device_a6b)
    cg_modules_md.load_state_dict(checkpoint['cg_modules'])
    nerve_ring_md.load_state_dict(checkpoint['nerve_ring'])
    coordinator_md.load_state_dict(checkpoint['coordinator'])
    history_a6b = checkpoint['history']
    start_epoch = checkpoint['epoch']
    print(f"Cargado desde epoch {start_epoch}, mejor acc: {max(history_a6b['test_acc'])*100:.2f}%")
else:
    print("Sin checkpoint, entrenando desde cero.")

best_a6b = max(history_a6b['test_acc']) if history_a6b['test_acc'] else 0.0

for epoch in range(start_epoch, EPOCHS_A6B):
    for mod in cg_modules_md: mod.train()
    nerve_ring_md.train(); coordinator_md.train()

    for imgs, labels in train_loader_a6b:
        imgs, labels = imgs.to(device_a6b), labels.to(device_a6b)

        # FF local loss
        opt_mod_md.zero_grad()
        ff_loss = sum(mod.ff_loss(imgs, labels) for mod in cg_modules_md)
        ff_loss.backward()
        opt_mod_md.step()

        # Coordinator con module dropout
        opt_coord_md.zero_grad()
        with torch.no_grad():
            bottlenecks = [mod(imgs) for mod in cg_modules_md]
            if random.random() < 0.5:
                drop_idx = random.randint(0, 3)
                bottlenecks[drop_idx] = torch.zeros_like(bottlenecks[drop_idx])
        h_ring, _ = nerve_ring_md(torch.stack(bottlenecks, dim=1))
        logits, _, _ = coordinator_md(h_ring)
        loss = criterion_md(logits, labels)
        loss.backward()
        opt_coord_md.step()

    sched_mod_md.step(); sched_coord_md.step()

    # Eval
    for mod in cg_modules_md: mod.eval()
    nerve_ring_md.eval(); coordinator_md.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a6b:
            imgs, labels = imgs.to(device_a6b), labels.to(device_a6b)
            bottlenecks = [mod(imgs) for mod in cg_modules_md]
            h_ring, _ = nerve_ring_md(torch.stack(bottlenecks, dim=1))
            preds, _, _ = coordinator_md(h_ring)
            preds = preds.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    history_a6b['test_acc'].append(acc)
    if acc > best_a6b:
        best_a6b = acc
        torch.save({
            'cg_modules': cg_modules_md.state_dict(),
            'nerve_ring': nerve_ring_md.state_dict(),
            'coordinator': coordinator_md.state_dict(),
            'history': history_a6b,
            'epoch': epoch + 1,
        }, f"{SAVE_DIR}/a6b_final.pt")
    print(f"  epoch {epoch+1}/{EPOCHS_A6B}  val_acc={acc:.4f}  best={best_a6b:.4f}")

print(f"\nA6b best: {best_a6b*100:.2f}%  |  A18b: 64.17%")

In [ ]:
# A6b — Resilience test: same protocol as A6 / A18b
from itertools import combinations

def eval_a6b(mods, nring, coord, loader, device, failed_indices=None):
    for m in mods: m.eval()
    nring.eval(); coord.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            bns = [m(imgs) for m in mods]
            if failed_indices:
                for fi in failed_indices:
                    bns[fi] = torch.zeros_like(bns[fi])
            h, _ = nring(torch.stack(bns, dim=1))
            preds, _, _ = coord(h)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

N = len(cg_modules_md)
print("A6b Resilience Test")
print("=" * 40)

# Full system
full_acc = eval_a6b(cg_modules_md, nerve_ring_md, coordinator_md, val_loader_a6b, device_a6b)
print(f"Full system: {full_acc*100:.2f}%")

# Single module failures
print("\nSingle module failure:")
single_accs = []
for i in range(N):
    acc = eval_a6b(cg_modules_md, nerve_ring_md, coordinator_md, val_loader_a6b, device_a6b, [i])
    single_accs.append(acc)
    print(f"  M{i} fails: {acc*100:.2f}%")
print(f"  Floor (worst): {min(single_accs)*100:.2f}%  |  A18b floor: 41.47%")

# Double module failures
print("\nDouble module failure:")
double_accs = []
for combo in combinations(range(N), 2):
    acc = eval_a6b(cg_modules_md, nerve_ring_md, coordinator_md, val_loader_a6b, device_a6b, list(combo))
    double_accs.append(acc)
    print(f"  M{combo[0]}+M{combo[1]} fail: {acc*100:.2f}%")
print(f"  Floor (worst): {min(double_accs)*100:.2f}%")

## 27. Experiment A18c: CGCNNModule + Multiscale

Misma arquitectura que A6b pero con entradas multi-resolución [32,16,8,4] por módulo.

**Pregunta**: ¿Fourier+multiscala suman sobre A6b (64.34%)? ¿O CG ya captura todo?

**Comparación**:
- A6b (CG + ModDrop, sin multiscala): 64.34%
- A18c (CG + ModDrop + multiscala): ?

In [ ]:
# A18c — CGCNNModule + Multiscale + Module Dropout
import torch, torch.nn as nn, torch.nn.functional as F, random
from torch.optim.lr_scheduler import MultiStepLR

device_a18c = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_A18C = 30
KERNEL_SIZES_A18C = [3, 5, 7, 9]
INPUT_SCALES_A18C = [32, 16, 8, 4]

cfg_a18c = OctopusNetConfig(
    dataset="cifar10", epochs=EPOCHS_A18C, batch_size=128,
    bottleneck_size=64, use_nerve_ring=True,
)
train_loader_a18c, val_loader_a18c = get_dataloaders(cfg_a18c)

cg_modules_a18c = nn.ModuleList([
    CGCNNModule(kernel_size=KERNEL_SIZES_A18C[i], bottleneck_size=64).to(device_a18c)
    for i in range(4)
])
nerve_ring_a18c  = NerveRing(bottleneck_size=64).to(device_a18c)
coordinator_a18c = Coordinator(num_modules=4, bottleneck_size=64, num_classes=10).to(device_a18c)

opt_mod_a18c   = torch.optim.Adam(cg_modules_a18c.parameters(), lr=1e-3)
opt_coord_a18c = torch.optim.Adam(
    list(nerve_ring_a18c.parameters()) + list(coordinator_a18c.parameters()), lr=1e-3
)
sched_mod_a18c   = MultiStepLR(opt_mod_a18c,   milestones=[10,20,27], gamma=0.1)
sched_coord_a18c = MultiStepLR(opt_coord_a18c, milestones=[10,20,27], gamma=0.1)
criterion_a18c = nn.CrossEntropyLoss()
history_a18c = {"val_acc": []}
best_a18c = 0.0

for epoch in range(EPOCHS_A18C):
    for mod in cg_modules_a18c: mod.train()
    nerve_ring_a18c.train(); coordinator_a18c.train()

    for imgs, labels in train_loader_a18c:
        imgs, labels = imgs.to(device_a18c), labels.to(device_a18c)

        opt_mod_a18c.zero_grad()
        ff_loss = sum(
            cg_modules_a18c[i].ff_loss(
                F.interpolate(imgs, size=(INPUT_SCALES_A18C[i], INPUT_SCALES_A18C[i]),
                              mode='bilinear', align_corners=False),
                labels
            ) for i in range(4)
        )
        ff_loss.backward()
        opt_mod_a18c.step()

        opt_coord_a18c.zero_grad()
        with torch.no_grad():
            bottlenecks = [
                cg_modules_a18c[i](
                    F.interpolate(imgs, size=(INPUT_SCALES_A18C[i], INPUT_SCALES_A18C[i]),
                                  mode='bilinear', align_corners=False)
                ) for i in range(4)
            ]
            if random.random() < 0.5:
                bottlenecks[random.randint(0,3)] = torch.zeros_like(bottlenecks[0])
        h_ring, _ = nerve_ring_a18c(torch.stack(bottlenecks, dim=1))
        logits, _, _ = coordinator_a18c(h_ring)
        loss = criterion_a18c(logits, labels)
        loss.backward()
        opt_coord_a18c.step()

    sched_mod_a18c.step(); sched_coord_a18c.step()

    for mod in cg_modules_a18c: mod.eval()
    nerve_ring_a18c.eval(); coordinator_a18c.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a18c:
            imgs, labels = imgs.to(device_a18c), labels.to(device_a18c)
            bottlenecks = [
                cg_modules_a18c[i](
                    F.interpolate(imgs, size=(INPUT_SCALES_A18C[i], INPUT_SCALES_A18C[i]),
                                  mode='bilinear', align_corners=False)
                ) for i in range(4)
            ]
            h_ring, _ = nerve_ring_a18c(torch.stack(bottlenecks, dim=1))
            preds, _, _ = coordinator_a18c(h_ring)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    history_a18c["val_acc"].append(acc)
    if acc > best_a18c:
        best_a18c = acc
    print(f"  epoch {epoch+1}/{EPOCHS_A18C}  val_acc={acc:.4f}  best={best_a18c:.4f}")

print(f"\nA18c: {best_a18c*100:.2f}%  |  A6b (sin multiscala): 64.34%")

## 28. Experiment A19: CGCNNModule + ResBlocks

ResBlocks internos con GroupNorm dentro de CGCNNModule. Misma config exacta que A6b.

In [ ]:
# ============================================================
# A19 -- CGCNNModule + ResBlocks (aislado, no modifica .py)
# Misma config exacta que A6b: 30 epochs, MultiStepLR, ModDrop p=0.5
# Solo cambia: CGCNNModule -> CGCNNModuleV2 (stem + 2 ResBlocks + GroupNorm)
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F, random
from torch.optim.lr_scheduler import MultiStepLR
from itertools import combinations

# Arquitectura A19
class ResBlockCG(nn.Module):
    def __init__(self, ch, groups):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, groups=groups),
            nn.GroupNorm(groups, ch),
            nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1, groups=groups),
            nn.GroupNorm(groups, ch),
        )
    def forward(self, x):
        return x + F.relu(self.block(x))

class CGCNNModuleV2(nn.Module):
    """A19: CGCNNModule + 2 ResBlocks internos con GroupNorm."""
    def __init__(self, num_classes=10, channels_per_group=16,
                 kernel_size=3, in_channels=3, bottleneck_size=64):
        super().__init__()
        self.num_classes = num_classes
        self.cpg = channels_per_group
        total_ch = num_classes * channels_per_group  # 160

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, total_ch, kernel_size, padding=kernel_size // 2),
            nn.GroupNorm(num_classes, total_ch),
            nn.ReLU(),
        )
        self.res1 = ResBlockCG(total_ch, groups=num_classes)
        self.res2 = ResBlockCG(total_ch, groups=num_classes)
        self.pool = nn.AdaptiveAvgPool2d(4)
        self.proj = nn.Linear(total_ch * 16, bottleneck_size)

    def forward(self, x):
        x = self.stem(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.proj(self.pool(x).flatten(1))

    def goodness_per_group(self, x):
        x = self.stem(x)
        x = self.res1(x)
        x = self.res2(x)
        feat = self.pool(x)
        B, C, H, W = feat.shape
        return feat.view(B, self.num_classes, self.cpg, H * W).pow(2).mean(dim=[2, 3])

    def ff_loss(self, x, labels, threshold=2.0):
        g = self.goodness_per_group(x)
        g_target = g[torch.arange(len(labels)), labels]
        mask = torch.ones_like(g, dtype=torch.bool)
        mask[torch.arange(len(labels)), labels] = False
        g_wrong = g[mask].view(len(labels), self.num_classes - 1).mean(dim=1)
        return F.softplus(-g_target + threshold).mean() + F.softplus(g_wrong - threshold).mean()

# Setup: misma config exacta que A6b
device_a19 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_A19 = 30
KERNEL_SIZES_A19 = [3, 5, 7, 9]

cfg_a19 = OctopusNetConfig(
    dataset="cifar10", epochs=EPOCHS_A19,
    batch_size=128, bottleneck_size=64, use_nerve_ring=True,
)
train_loader_a19, val_loader_a19 = get_dataloaders(cfg_a19)

cg_modules_a19 = nn.ModuleList([
    CGCNNModuleV2(kernel_size=k, bottleneck_size=64).to(device_a19)
    for k in KERNEL_SIZES_A19
])
nerve_ring_a19  = NerveRing(bottleneck_size=64).to(device_a19)
coordinator_a19 = Coordinator(num_modules=4, bottleneck_size=64, num_classes=10).to(device_a19)

opt_mod_a19   = torch.optim.Adam(cg_modules_a19.parameters(), lr=1e-3)
opt_coord_a19 = torch.optim.Adam(
    list(nerve_ring_a19.parameters()) + list(coordinator_a19.parameters()), lr=1e-3
)
sched_mod_a19   = MultiStepLR(opt_mod_a19,   milestones=[10, 20, 27], gamma=0.1)
sched_coord_a19 = MultiStepLR(opt_coord_a19, milestones=[10, 20, 27], gamma=0.1)
criterion_a19 = nn.CrossEntropyLoss()
history_a19 = {"val_acc": []}
best_a19 = 0.0
MODULE_DROPOUT_A19 = 0.5

print("A19 setup listo")

# Training loop
for epoch in range(EPOCHS_A19):
    for mod in cg_modules_a19: mod.train()
    nerve_ring_a19.train(); coordinator_a19.train()

    for imgs, labels in train_loader_a19:
        imgs, labels = imgs.to(device_a19), labels.to(device_a19)

        opt_mod_a19.zero_grad()
        ff_loss = sum(m.ff_loss(imgs, labels) for m in cg_modules_a19)
        ff_loss.backward()
        opt_mod_a19.step()

        opt_coord_a19.zero_grad()
        with torch.no_grad():
            bns = [m(imgs) for m in cg_modules_a19]
            if random.random() < MODULE_DROPOUT_A19:
                bns[random.randint(0, 3)] = torch.zeros_like(bns[0])
            h, _ = nerve_ring_a19(torch.stack(bns, dim=1))
        logits, _, _ = coordinator_a19(h)
        loss = criterion_a19(logits, labels)
        loss.backward()
        opt_coord_a19.step()

    sched_mod_a19.step()
    sched_coord_a19.step()

    for mod in cg_modules_a19: mod.eval()
    nerve_ring_a19.eval(); coordinator_a19.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a19:
            imgs, labels = imgs.to(device_a19), labels.to(device_a19)
            bns = [m(imgs) for m in cg_modules_a19]
            h, _ = nerve_ring_a19(torch.stack(bns, dim=1))
            preds, _, _ = coordinator_a19(h)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    history_a19["val_acc"].append(val_acc)
    if val_acc > best_a19:
        best_a19 = val_acc
    print(f"  epoch {epoch+1}/{EPOCHS_A19}  val_acc={val_acc:.4f}  best={best_a19:.4f}")

print(f"\nA19 best: {best_a19*100:.2f}%  |  A6b: 64.34%")

# Resilience test
def eval_a19(mods, nring, coord, loader, device, failed=None):
    for m in mods: m.eval()
    nring.eval(); coord.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            bns = [m(imgs) for m in mods]
            if failed:
                for fi in failed:
                    bns[fi] = torch.zeros_like(bns[fi])
            h, _ = nring(torch.stack(bns, dim=1))
            preds, _, _ = coord(h)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

print("\nA19 Resilience Test")
print("=" * 40)
full = eval_a19(cg_modules_a19, nerve_ring_a19, coordinator_a19, val_loader_a19, device_a19)
print(f"Full system: {full*100:.2f}%")

print("\nSingle failure:")
single = []
for i in range(4):
    acc = eval_a19(cg_modules_a19, nerve_ring_a19, coordinator_a19, val_loader_a19, device_a19, [i])
    single.append(acc)
    print(f"  M{i} falla: {acc*100:.2f}%")
print(f"  Floor: {min(single)*100:.2f}%  |  A6b floor: 61.12%")

print("\nDouble failure:")
double = []
for combo in combinations(range(4), 2):
    acc = eval_a19(cg_modules_a19, nerve_ring_a19, coordinator_a19, val_loader_a19, device_a19, list(combo))
    double.append(acc)
    print(f"  M{list(combo)} fallan: {acc*100:.2f}%")
print(f"  Floor doble: {min(double)*100:.2f}%  |  A6b floor doble: 52.87%")


## 29. Experiment A20: CGCNNModule + Larger Pool

Misma config que A6b. Una sola variable cambia: AdaptiveAvgPool2d(4) -> (6). Hipotesis: pooling agresivo era el cuello de botella en A19.

In [ ]:
# ============================================================
# A20 -- CGCNNModule + AdaptiveAvgPool2d(6) (aislado, no modifica .py)
# Misma config exacta que A6b. Solo cambia: pool 4x4 -> 6x6
# proj ajustado: Linear(total_ch * 36, bottleneck_size)
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F, random
from torch.optim.lr_scheduler import MultiStepLR
from itertools import combinations


class CGCNNModuleV3(nn.Module):
    """A20: CGCNNModule con AdaptiveAvgPool2d(6) en vez de (4)."""
    def __init__(self, num_classes=10, channels_per_group=16,
                 kernel_size=3, in_channels=3, bottleneck_size=64, pool_size=6):
        super().__init__()
        self.num_classes = num_classes
        self.cpg = channels_per_group
        self.pool_size = pool_size
        total_ch = num_classes * channels_per_group  # 160

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, total_ch, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.Conv2d(total_ch, total_ch, kernel_size, padding=kernel_size // 2,
                      groups=num_classes),
            nn.BatchNorm2d(total_ch),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(pool_size),
        )
        self.proj = nn.Linear(total_ch * pool_size * pool_size, bottleneck_size)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

    def goodness_per_group(self, x):
        feat = self.conv(x)
        B, C, H, W = feat.shape
        return feat.view(B, self.num_classes, self.cpg, H * W).pow(2).mean(dim=[2, 3])

    def ff_loss(self, x, labels, threshold=2.0):
        g = self.goodness_per_group(x)
        g_target = g[torch.arange(len(labels)), labels]
        mask = torch.ones_like(g, dtype=torch.bool)
        mask[torch.arange(len(labels)), labels] = False
        g_wrong = g[mask].view(len(labels), self.num_classes - 1).mean(dim=1)
        return F.softplus(-g_target + threshold).mean() + F.softplus(g_wrong - threshold).mean()


# Setup: misma config exacta que A6b
device_a20 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_A20 = 30
KERNEL_SIZES_A20 = [3, 5, 7, 9]
POOL_SIZE_A20 = 6

cfg_a20 = OctopusNetConfig(
    dataset="cifar10", epochs=EPOCHS_A20,
    batch_size=128, bottleneck_size=64, use_nerve_ring=True,
)
train_loader_a20, val_loader_a20 = get_dataloaders(cfg_a20)

cg_modules_a20 = nn.ModuleList([
    CGCNNModuleV3(kernel_size=k, bottleneck_size=64, pool_size=POOL_SIZE_A20).to(device_a20)
    for k in KERNEL_SIZES_A20
])
nerve_ring_a20  = NerveRing(bottleneck_size=64).to(device_a20)
coordinator_a20 = Coordinator(num_modules=4, bottleneck_size=64, num_classes=10).to(device_a20)

opt_mod_a20   = torch.optim.Adam(cg_modules_a20.parameters(), lr=1e-3)
opt_coord_a20 = torch.optim.Adam(
    list(nerve_ring_a20.parameters()) + list(coordinator_a20.parameters()), lr=1e-3
)
sched_mod_a20   = MultiStepLR(opt_mod_a20,   milestones=[10, 20, 27], gamma=0.1)
sched_coord_a20 = MultiStepLR(opt_coord_a20, milestones=[10, 20, 27], gamma=0.1)
criterion_a20 = nn.CrossEntropyLoss()
history_a20 = {"val_acc": []}
best_a20 = 0.0
MODULE_DROPOUT_A20 = 0.5

total_params = sum(p.numel() for m in cg_modules_a20 for p in m.parameters())
print(f"A20 setup listo -- {total_params:,} params en modulos (pool={POOL_SIZE_A20}x{POOL_SIZE_A20})")

# Training loop
for epoch in range(EPOCHS_A20):
    for mod in cg_modules_a20: mod.train()
    nerve_ring_a20.train(); coordinator_a20.train()

    for imgs, labels in train_loader_a20:
        imgs, labels = imgs.to(device_a20), labels.to(device_a20)

        opt_mod_a20.zero_grad()
        ff_loss = sum(m.ff_loss(imgs, labels) for m in cg_modules_a20)
        ff_loss.backward()
        opt_mod_a20.step()

        opt_coord_a20.zero_grad()
        with torch.no_grad():
            bns = [m(imgs) for m in cg_modules_a20]
            if random.random() < MODULE_DROPOUT_A20:
                bns[random.randint(0, 3)] = torch.zeros_like(bns[0])
            h, _ = nerve_ring_a20(torch.stack(bns, dim=1))
        logits, _, _ = coordinator_a20(h)
        loss = criterion_a20(logits, labels)
        loss.backward()
        opt_coord_a20.step()

    sched_mod_a20.step()
    sched_coord_a20.step()

    for mod in cg_modules_a20: mod.eval()
    nerve_ring_a20.eval(); coordinator_a20.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader_a20:
            imgs, labels = imgs.to(device_a20), labels.to(device_a20)
            bns = [m(imgs) for m in cg_modules_a20]
            h, _ = nerve_ring_a20(torch.stack(bns, dim=1))
            preds, _, _ = coordinator_a20(h)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    history_a20["val_acc"].append(val_acc)
    if val_acc > best_a20:
        best_a20 = val_acc
    print(f"  epoch {epoch+1}/{EPOCHS_A20}  val_acc={val_acc:.4f}  best={best_a20:.4f}")

print(f"\nA20 best: {best_a20*100:.2f}%  |  A6b: 64.34%")

# Resilience test
def eval_a20(mods, nring, coord, loader, device, failed=None):
    for m in mods: m.eval()
    nring.eval(); coord.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            bns = [m(imgs) for m in mods]
            if failed:
                for fi in failed:
                    bns[fi] = torch.zeros_like(bns[fi])
            h, _ = nring(torch.stack(bns, dim=1))
            preds, _, _ = coord(h)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

print("\nA20 Resilience Test")
print("=" * 40)
full = eval_a20(cg_modules_a20, nerve_ring_a20, coordinator_a20, val_loader_a20, device_a20)
print(f"Full system: {full*100:.2f}%")

print("\nSingle failure:")
single = []
for i in range(4):
    acc = eval_a20(cg_modules_a20, nerve_ring_a20, coordinator_a20, val_loader_a20, device_a20, [i])
    single.append(acc)
    print(f"  M{i} falla: {acc*100:.2f}%")
print(f"  Floor: {min(single)*100:.2f}%  |  A6b floor: 61.12%")

print("\nDouble failure:")
double = []
for combo in combinations(range(4), 2):
    acc = eval_a20(cg_modules_a20, nerve_ring_a20, coordinator_a20, val_loader_a20, device_a20, list(combo))
    double.append(acc)
    print(f"  M{list(combo)} fallan: {acc*100:.2f}%")
print(f"  Floor doble: {min(double)*100:.2f}%  |  A6b floor doble: 52.87%")
